# EXP 04A — CatBoost Objective Refinement, Tail/Blowout-Aware Targets, dan Freeze-Safe Decoder untuk History-Full Backbone

Notebook ini adalah **refinement langsung** terhadap backbone terbaik dari eksperimen sebelumnya, terutama jalur **history-full + freeze-safe** yang sudah terbukti paling stabil.

Fokus EXP04A bukan membuka cabang baru, melainkan memperbaiki dua bottleneck yang sekarang terlihat paling penting:

1. **objective model goal** agar lebih cocok untuk target count / skewed goals, dan  
2. **decoder** agar lebih peka terhadap pertandingan tail / blowout tanpa merusak performa umum di match normal.

Eksperimen ini **standalone**. Notebook tidak membaca output file eksperimen lama sebagai dependency.  
Input hanya berasal dari:

- `data/train.csv`
- `data/test.csv`
- `data/sample submission.csv`
- `data/metadata.txt`

Metrik utama yang dipakai tetap **AW-MAE**.


## 01. Setup, Seed, Path, dan Warning Eksperimen

Di tahap ini kita menyiapkan:

- import library,
- seed global,
- path project,
- output folder eksperimen,
- dan guardrail bahwa notebook ini tetap **freeze-safe only**.

Eksperimen ini sengaja **tidak** membuka branch recursive, probabilistic full branch, distillation, atau ensemble besar.


In [1]:
import os
import json
import math
import copy
import random
import warnings
from pathlib import Path
from collections import Counter, deque
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, mean_absolute_error

from catboost import CatBoostRegressor, CatBoostClassifier, Pool

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

# === KONFIGURASI GLOBAL ===
SEED = 42
EXPERIMENT_NAME = "exp04a_catboost_objective_tail_refinement"

# Resolve project root secara aman, supaya notebook bisa dijalankan dari root project atau dari folder notebook/
cwd = Path.cwd().resolve()
project_candidates = [cwd, cwd.parent, cwd.parent.parent]
PROJECT_ROOT = None
for cand in project_candidates:
    if (cand / "data").exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / EXPERIMENT_NAME
FIG_DIR = OUTPUT_ROOT / "figures"
PRED_DIR = OUTPUT_ROOT / "predictions"
SUB_DIR = OUTPUT_ROOT / "submissions"
SUM_DIR = OUTPUT_ROOT / "summaries"

for d in [NOTEBOOK_DIR, OUTPUT_ROOT, FIG_DIR, PRED_DIR, SUB_DIR, SUM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def _resolve_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]

TRAIN_PATH = _resolve_existing_path([DATA_DIR / "train.csv"])
TEST_PATH = _resolve_existing_path([DATA_DIR / "test.csv"])
SAMPLE_SUB_PATH = _resolve_existing_path([
    DATA_DIR / "sample submission.csv",
    DATA_DIR / "sample_submission.csv",
])
METADATA_PATH = _resolve_existing_path([DATA_DIR / "metadata.txt"])

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR      : {DATA_DIR}")
print(f"OUTPUT_ROOT   : {OUTPUT_ROOT}")


PROJECT_ROOT : D:\Github\gamafest-bcc-lagi-bawa-anak-baru
DATA_DIR      : D:\Github\gamafest-bcc-lagi-bawa-anak-baru\data
OUTPUT_ROOT   : D:\Github\gamafest-bcc-lagi-bawa-anak-baru\outputs\exp04a_catboost_objective_tail_refinement


In [2]:
def seed_everything(seed: int = 42) -> None:
    """Set seed dasar untuk reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)
print(f"[OK] Seed aktif: {SEED}")


[OK] Seed aktif: 42


### Warning eksperimen

- Notebook ini **standalone**.
- Notebook ini **tidak membaca output eksperimen lama** sebagai dependency.
- Jalur inference yang dipakai adalah **freeze-safe only**.
- Analisis GT parsial dari luar notebook hanya dipakai sebagai **insight konseptual manusia**, bukan sebagai input eksekusi.
- Semua tuning decoder dilakukan dengan prinsip **raw-output-first**.


## 02. Validasi File Input

Sebelum masuk ke pipeline utama, kita validasi dulu file input yang wajib ada.


In [3]:
def validate_input_files(file_paths: dict) -> None:
    missing = []
    print("=== INPUT FILE CHECK ===")
    for name, path in file_paths.items():
        exists = Path(path).exists()
        size_mb = Path(path).stat().st_size / (1024 * 1024) if exists else np.nan
        print(f"- {name:<12}: {path} | exists={exists} | size_mb={size_mb:.4f}" if exists else f"- {name:<12}: {path} | exists={exists}")
        if not exists:
            missing.append(str(path))
    if missing:
        raise FileNotFoundError(f"Masih ada file input yang belum ditemukan: {missing}")

FILE_PATHS = {
    "train": TRAIN_PATH,
    "test": TEST_PATH,
    "sample_sub": SAMPLE_SUB_PATH,
    "metadata": METADATA_PATH,
}
validate_input_files(FILE_PATHS)


=== INPUT FILE CHECK ===
- train       : D:\Github\gamafest-bcc-lagi-bawa-anak-baru\data\train.csv | exists=True | size_mb=22.4378
- test        : D:\Github\gamafest-bcc-lagi-bawa-anak-baru\data\test.csv | exists=True | size_mb=6.9827
- sample_sub  : D:\Github\gamafest-bcc-lagi-bawa-anak-baru\data\sample submission.csv | exists=True | size_mb=0.9055
- metadata    : D:\Github\gamafest-bcc-lagi-bawa-anak-baru\data\metadata.txt | exists=True | size_mb=0.0018


## 03. Load Data dan Helper Dasar

Di bagian ini kita load seluruh input dan membawa kembali helper inti yang dibutuhkan dari fondasi eksperimen sebelumnya:

- canonical match builder,
- evaluator AW-MAE,
- mapping balik ke submission,
- dan temporal holdout leakage-safe.


In [4]:
train = pd.read_csv(TRAIN_PATH, parse_dates=["date"])
test = pd.read_csv(TEST_PATH, parse_dates=["date"])
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
metadata_text = Path(METADATA_PATH).read_text(encoding="utf-8")

print(f"train shape      : {train.shape}")
print(f"test shape       : {test.shape}")
print(f"sample_sub shape : {sample_sub.shape}")
print("\n=== metadata preview ===")
print(metadata_text[:800])


train shape      : (78772, 47)
test shape       : (42422, 20)
sample_sub shape : (42422, 3)

=== metadata preview ===
1. Identitas & Info Dasar
Id: Identitas unik untuk setiap baris (Format: match_id_nama_tim).
match_id: ID unik untuk satu pertandingan (satu pertandingan memiliki dua baris Id untuk masing-masing tim).
date: Tanggal pertandingan dilaksanakan.
gender: Jenis kelamin pemain (M untuk Pria, W untuk Wanita).
team: Nama tim utama.
opponent: Nama tim lawan.

2. Kondisi Pertandingan
is_home: Binary (1/0), apakah tim bermain di kandang sendiri.
neutral: Binary (1/0), apakah pertandingan dimainkan di tempat netral.
tournament: Nama kompetisi atau jenis turnamen (misal: Friendly, FIFA World Cup, dll).
venue_country: Negara tempat pertandingan berlangsung.
confederation_team/opp: Konfederasi sepak bola tim/lawan (misal: UEFA, CAF, CONMEBOL).

3. Metrik Performa (Hanya tersedia lengkap di train.csv)
elo


In [5]:
# ============================================================
# HELPER FUNCTIONS — standalone dari EXP 00 / EXP 01 / EXP 02.1 / EXP 03
# ============================================================

EXACT_PENALTY = 0.30
OUTCOME_PENALTY = 0.25
GD_PENALTY = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER = 1.50

K_ELO = 24.0
K_GD = 6.0
EWMA_ALPHA = 0.35
ELO_HOME_BONUS = 60.0
GD_HOME_BONUS = 10.0


def _safe_ratio(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    out = a / b.replace(0, np.nan)
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def _deque_mean(dq, default=np.nan):
    return float(sum(dq) / len(dq)) if len(dq) > 0 else default


def _outcome(a: int, b: int) -> int:
    if a > b:
        return 1
    if a < b:
        return -1
    return 0


def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).lower().strip()
    if "fifa world cup" in t or t == "world cup":
        return 2.00
    if "afc championship" in t or "afc asian cup" in t or "asian cup" in t:
        return 1.80
    if "friendly" in t:
        return 0.96
    return 1.20


def official_match_loss(
    y_team_true: int,
    y_opp_true: int,
    y_team_pred: int,
    y_opp_pred: int,
) -> float:
    y_team_true = int(y_team_true)
    y_opp_true = int(y_opp_true)
    y_team_pred = int(y_team_pred)
    y_opp_pred = int(y_opp_pred)

    mae = (abs(y_team_true - y_team_pred) + abs(y_opp_true - y_opp_pred)) / 2.0
    exact = int((y_team_true == y_team_pred) and (y_opp_true == y_opp_pred))
    outcome = int(_outcome(y_team_true, y_opp_true) == _outcome(y_team_pred, y_opp_pred))
    gd = int((y_team_true - y_opp_true) == (y_team_pred - y_opp_pred))

    penalty = (
        EXACT_PENALTY * (1 - exact)
        + OUTCOME_PENALTY * (1 - outcome)
        + GD_PENALTY * (1 - gd)
    )
    multiplier = 1.0 if outcome == 1 else WRONG_OUTCOME_MULTIPLIER
    raw_loss = mae + penalty
    return float((raw_loss * multiplier) ** NONLINEAR_POWER)


def awmae_score(
    y_team_true,
    y_opp_true,
    y_team_pred,
    y_opp_pred,
    tournaments,
) -> float:
    y_team_true = np.asarray(y_team_true)
    y_opp_true = np.asarray(y_opp_true)
    y_team_pred = np.asarray(y_team_pred)
    y_opp_pred = np.asarray(y_opp_pred)
    tournaments = np.asarray(tournaments)

    if not (len(y_team_true) == len(y_opp_true) == len(y_team_pred) == len(y_opp_pred) == len(tournaments)):
        raise ValueError("Input awmae_score memiliki panjang yang tidak konsisten.")

    weights = np.array([get_tournament_weight(t) for t in tournaments], dtype=float)
    losses = np.array([
        official_match_loss(a, b, c, d)
        for a, b, c, d in zip(y_team_true, y_opp_true, y_team_pred, y_opp_pred)
    ], dtype=float)
    return float(np.sum(losses * weights) / np.sum(weights))


def make_time_based_holdout(
    train_match: pd.DataFrame,
    valid_fraction: float = 0.2,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "date" not in train_match.columns:
        raise KeyError("Kolom 'date' wajib ada untuk time-based split.")
    temp = train_match.sort_values(["date", "match_id"]).reset_index(drop=True)
    n_valid = max(1, int(math.ceil(len(temp) * valid_fraction)))
    n_valid = min(n_valid, max(1, len(temp) - 1))
    split_idx = len(temp) - n_valid
    train_fold = temp.iloc[:split_idx].reset_index(drop=True)
    valid_fold = temp.iloc[split_idx:].reset_index(drop=True)
    return train_fold, valid_fold


def build_match_level(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    """Konversi row-level ke match-level canonical.

    Canonical rule: team_a = alfabet pertama dalam satu match_id.
    """
    required = ["match_id", "team", "date"]
    for col in required:
        if col not in df.columns:
            raise KeyError(f"Kolom wajib tidak ditemukan: {col}")

    counts = df.groupby("match_id").size()
    bad = counts[counts != 2]
    if len(bad) > 0:
        raise ValueError(f"Ditemukan {len(bad)} match_id tanpa tepat dua row.")

    sort_cols = [c for c in ["match_id", "team", "Id"] if c in df.columns]
    temp = df.sort_values(sort_cols).reset_index(drop=True)

    row_a = temp.groupby("match_id", sort=False).nth(0).reset_index()
    row_b = temp.groupby("match_id", sort=False).nth(1).reset_index()

    if not (row_a["match_id"].values == row_b["match_id"].values).all():
        raise AssertionError("Pairing match-level gagal: urutan match_id tidak sinkron.")

    result = pd.DataFrame()
    scalar_cols = [
        "match_id", "date", "gender", "tournament", "venue_country", "neutral",
        "altitude_venue", "temperature_venue",
    ]
    for col in scalar_cols:
        if col in temp.columns:
            result[col] = row_a[col].values

    result["team_a"] = row_a["team"].astype(str).values
    result["team_b"] = row_b["team"].astype(str).values

    if "is_home" in temp.columns:
        result["team_a_is_home"] = pd.to_numeric(row_a["is_home"], errors="coerce").fillna(0).astype(float).values
        result["team_b_is_home"] = pd.to_numeric(row_b["is_home"], errors="coerce").fillna(0).astype(float).values
    else:
        result["team_a_is_home"] = 0.0
        result["team_b_is_home"] = 0.0

    side_map = {
        "confederation_team": "confederation",
        "population_team": "population",
        "gdp_per_capita_team": "gdp_per_capita",
        "distance_travel_team": "distance_travel",
    }
    for raw_col, suffix in side_map.items():
        if raw_col in temp.columns:
            result[f"team_a_{suffix}"] = row_a[raw_col].values
            result[f"team_b_{suffix}"] = row_b[raw_col].values
        else:
            result[f"team_a_{suffix}"] = np.nan
            result[f"team_b_{suffix}"] = np.nan

    if is_train:
        if "team_goals" not in temp.columns or "opp_goals" not in temp.columns:
            raise KeyError("Target team_goals/opp_goals tidak ditemukan di train.")

        team_a_goals = pd.to_numeric(row_a["team_goals"], errors="coerce").astype(float).values
        team_b_goals = pd.to_numeric(row_a["opp_goals"], errors="coerce").astype(float).values

        # Optional consistency check vs row_b
        row_b_team_goals = pd.to_numeric(row_b["team_goals"], errors="coerce").astype(float).values
        row_b_opp_goals = pd.to_numeric(row_b["opp_goals"], errors="coerce").astype(float).values

        if not np.allclose(team_a_goals, row_b_opp_goals, equal_nan=True):
            raise AssertionError("Inkonsistensi target: team_a_goals != row_b.opp_goals")
        if not np.allclose(team_b_goals, row_b_team_goals, equal_nan=True):
            raise AssertionError("Inkonsistensi target: team_b_goals != row_b.team_goals")

        result["team_a_goals"] = team_a_goals
        result["team_b_goals"] = team_b_goals

    return result


def match_predictions_to_submission(
    test_row_df: pd.DataFrame,
    pred_match_df: pd.DataFrame,
    canonical_team_a_col: str = "team_a",
    canonical_team_b_col: str = "team_b",
    pred_a_col: str = "pred_team_a_goals",
    pred_b_col: str = "pred_team_b_goals",
) -> pd.DataFrame:
    need_cols = ["match_id", canonical_team_a_col, canonical_team_b_col, pred_a_col, pred_b_col]
    missing = [c for c in need_cols if c not in pred_match_df.columns]
    if missing:
        raise KeyError(f"Kolom pred_match_df belum lengkap: {missing}")

    tmp = test_row_df[["Id", "match_id", "team"]].copy()
    merged = tmp.merge(pred_match_df[need_cols], on="match_id", how="left")

    is_a = merged["team"].astype(str) == merged[canonical_team_a_col].astype(str)
    merged["team_goals"] = np.where(is_a, merged[pred_a_col], merged[pred_b_col])
    merged["opp_goals"] = np.where(is_a, merged[pred_b_col], merged[pred_a_col])

    return merged[["Id", "team_goals", "opp_goals"]].copy()


print("[OK] Helper dasar siap.")


[OK] Helper dasar siap.


## 04. Cleaning Awal dan Canonical Match Data

Tahap ini melakukan pembersihan konservatif, lalu membangun representasi **match-level canonical** yang stabil untuk train dan test.

Pembersihan yang dilakukan:

- `altitude_venue == -9999` dianggap missing,
- string column dirapikan,
- `date` dipastikan valid,
- lalu row-level diubah menjadi match-level.


In [6]:
def _clean_row_level_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "altitude_venue" in out.columns:
        out["altitude_venue"] = pd.to_numeric(out["altitude_venue"], errors="coerce")
        out.loc[out["altitude_venue"] == -9999, "altitude_venue"] = np.nan

    if "temperature_venue" in out.columns:
        out["temperature_venue"] = pd.to_numeric(out["temperature_venue"], errors="coerce")

    if "neutral" in out.columns:
        out["neutral"] = pd.to_numeric(out["neutral"], errors="coerce").fillna(0).clip(0, 1)

    string_cols = out.select_dtypes(include="object").columns.tolist()
    for col in string_cols:
        out[col] = (
            out[col]
            .fillna("UNKNOWN")
            .astype(str)
            .str.strip()
            .replace({"": "UNKNOWN"})
        )

    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out.sort_values(["date", "match_id"]).reset_index(drop=True)
    return out

train_clean = _clean_row_level_df(train)
test_clean = _clean_row_level_df(test)

train_match_base = build_match_level(train_clean, is_train=True).sort_values(["date", "match_id"]).reset_index(drop=True)
test_match_base = build_match_level(test_clean, is_train=False).sort_values(["date", "match_id"]).reset_index(drop=True)

print(f"train_match_base shape : {train_match_base.shape}")
print(f"test_match_base shape  : {test_match_base.shape}")
display(train_match_base.head(3))


train_match_base shape : (39386, 22)
test_match_base shape  : (21211, 20)


,match_id,date,gender,tournament,venue_country,neutral,altitude_venue,temperature_venue,team_a,team_b,team_a_is_home,team_b_is_home,team_a_confederation,team_b_confederation,team_a_population,team_b_population,team_a_gdp_per_capita,team_b_gdp_per_capita,team_a_distance_travel,team_b_distance_travel,team_a_goals,team_b_goals
0,M000001,1872-11-30,M,Friendly,Scotland,0,NaN,NaN,England,Scotland,0.0,1.0,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
1,M000002,1873-03-08,M,Friendly,England,0,NaN,NaN,England,Scotland,1.0,0.0,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,4.0,2.0
2,M000003,1874-03-07,M,Friendly,Scotland,0,NaN,NaN,England,Scotland,0.0,1.0,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0


## 05. Static + History-Full Feature Pipeline

Seperti eksperimen terbaik sebelumnya, kita tetap mempertahankan dua blok fitur utama:

1. **static shared features**
2. **history-full legal features**

Bagian EXP04A ini sengaja **tidak membuka cabang fitur baru**.  
Kita hanya me-rebuild backbone yang sudah terbukti stabil.


In [7]:
def engineer_static_match_features(df_match: pd.DataFrame) -> pd.DataFrame:
    """Bangun block static shared features yang konsisten dengan branch sebelumnya."""
    df = df_match.copy()
    dt = pd.to_datetime(df["date"], errors="coerce")

    # Date features
    df["match_year"] = dt.dt.year
    df["match_month"] = dt.dt.month
    df["match_quarter"] = dt.dt.quarter
    df["match_dayofweek"] = dt.dt.dayofweek
    df["match_dayofyear"] = dt.dt.dayofyear
    df["match_is_weekend"] = (dt.dt.dayofweek >= 5).astype(float)
    df["match_decade"] = (dt.dt.year // 10) * 10

    # Venue / tournament / context
    df["home_side"] = np.where(
        pd.to_numeric(df["team_a_is_home"], errors="coerce").fillna(0) > 0,
        1,
        np.where(pd.to_numeric(df["team_b_is_home"], errors="coerce").fillna(0) > 0, -1, 0),
    ).astype(float)

    conf_a = df["team_a_confederation"].fillna("UNKNOWN").astype(str)
    conf_b = df["team_b_confederation"].fillna("UNKNOWN").astype(str)
    df["same_confederation"] = (conf_a == conf_b).astype(float)

    tournament_low = df["tournament"].fillna("").astype(str).str.lower()
    df["is_friendly"] = tournament_low.str.contains("friendly").astype(float)
    df["is_world_cup"] = tournament_low.str.contains("world cup").astype(float)
    df["is_qualification"] = tournament_low.str.contains("qualif").astype(float)
    df["is_nations_league"] = tournament_low.str.contains("nations league").astype(float)
    df["tournament_weight_proxy"] = df["tournament"].apply(get_tournament_weight)

    # Symmetry-aware numeric block
    pairs = [
        ("population", "team_a_population", "team_b_population"),
        ("gdp", "team_a_gdp_per_capita", "team_b_gdp_per_capita"),
        ("distance", "team_a_distance_travel", "team_b_distance_travel"),
    ]
    for prefix, col_a, col_b in pairs:
        a = pd.to_numeric(df[col_a], errors="coerce")
        b = pd.to_numeric(df[col_b], errors="coerce")

        df[f"{prefix}_a"] = a
        df[f"{prefix}_b"] = b
        df[f"{prefix}_diff"] = a - b
        df[f"{prefix}_abs_diff"] = (a - b).abs()
        df[f"log_{prefix}_a"] = np.log1p(a.clip(lower=0))
        df[f"log_{prefix}_b"] = np.log1p(b.clip(lower=0))
        df[f"log_{prefix}_diff"] = df[f"log_{prefix}_a"] - df[f"log_{prefix}_b"]
        df[f"{prefix}_ratio_ab"] = _safe_ratio(a, b)

    df["altitude_venue"] = pd.to_numeric(df["altitude_venue"], errors="coerce")
    df["temperature_venue"] = pd.to_numeric(df["temperature_venue"], errors="coerce")

    # Pair keys
    df["pair_key"] = df["team_a"].astype(str) + "__VS__" + df["team_b"].astype(str)
    df["confed_pair_key"] = conf_a + "__VS__" + conf_b

    return df


train_static_df = engineer_static_match_features(train_match_base)
test_static_df = engineer_static_match_features(test_match_base)

print(f"train_static_df shape : {train_static_df.shape}")
print(f"test_static_df shape  : {test_static_df.shape}")
display(train_static_df.head(2))


train_static_df shape : (39386, 62)
test_static_df shape  : (21211, 60)


,match_id,date,gender,tournament,venue_country,neutral,altitude_venue,temperature_venue,team_a,team_b,team_a_is_home,team_b_is_home,team_a_confederation,team_b_confederation,team_a_population,team_b_population,team_a_gdp_per_capita,team_b_gdp_per_capita,team_a_distance_travel,team_b_distance_travel,team_a_goals,team_b_goals,match_year,match_month,match_quarter,match_dayofweek,match_dayofyear,match_is_weekend,match_decade,home_side,same_confederation,is_friendly,is_world_cup,is_qualification,is_nations_league,tournament_weight_proxy,population_a,population_b,population_diff,population_abs_diff,log_population_a,log_population_b,log_population_diff,population_ratio_ab,gdp_a,gdp_b,gdp_diff,gdp_abs_diff,log_gdp_a,log_gdp_b,log_gdp_diff,gdp_ratio_ab,distance_a,distance_b,distance_diff,distance_abs_diff,log_distance_a,log_distance_b,log_distance_diff,distance_ratio_ab,pair_key,confed_pair_key
0,M000001,1872-11-30,M,Friendly,Scotland,0,NaN,NaN,England,Scotland,0.0,1.0,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1872,11,4,5,335,1.0,1870,-1.0,1.0,1.0,0.0,0.0,0.0,0.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,England__VS__Scotland,UEFA__VS__UEFA
1,M000002,1873-03-08,M,Friendly,England,0,NaN,NaN,England,Scotland,1.0,0.0,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,4.0,2.0,1873,3,1,5,67,1.0,1870,1.0,1.0,1.0,0.0,0.0,0.0,0.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,England__VS__Scotland,UEFA__VS__UEFA


In [8]:
def init_team_state() -> dict:
    return {
        "matches_played": 0,
        "last_match_date": pd.NaT,
        "elo_overall": 1500.0,
        "elo_goal_diff": 0.0,
        "ewm_points": 1.0,
        "ewm_goals_for": 1.2,
        "ewm_goals_against": 1.2,
        "ewm_goal_diff": 0.0,
        "recent_points_5": deque(maxlen=5),
        "recent_points_10": deque(maxlen=10),
        "recent_gf_5": deque(maxlen=5),
        "recent_ga_5": deque(maxlen=5),
        "recent_gd_5": deque(maxlen=5),
        "recent_results_10": deque(maxlen=10),
        "recent_clean_sheet_5": deque(maxlen=5),
        "recent_failed_to_score_5": deque(maxlen=5),
    }


def init_h2h_state() -> dict:
    return {
        "h2h_matches_played": 0,
        "h2h_points_a_last3": deque(maxlen=3),
        "h2h_gd_a_last3": deque(maxlen=3),
        "h2h_total_goals_last3": deque(maxlen=3),
    }


def expected_elo_result(rating_a: float, rating_b: float, home_bonus: float = 0.0) -> float:
    return 1.0 / (1.0 + 10 ** (-(rating_a + home_bonus - rating_b) / 400.0))


def snapshot_team_features_full(state: dict, match_date: pd.Timestamp) -> dict:
    last_date = state["last_match_date"]
    if pd.isna(last_date) or pd.isna(match_date):
        days_since = np.nan
    else:
        days_since = float((pd.Timestamp(match_date) - pd.Timestamp(last_date)).days)

    return {
        "hist_matches_played": float(state["matches_played"]),
        "hist_elo_overall": float(state["elo_overall"]),
        "hist_elo_gd": float(state["elo_goal_diff"]),
        "hist_ewm_points": float(state["ewm_points"]),
        "hist_ewm_gf": float(state["ewm_goals_for"]),
        "hist_ewm_ga": float(state["ewm_goals_against"]),
        "hist_ewm_gd": float(state["ewm_goal_diff"]),
        "hist_points_avg_last5": _deque_mean(state["recent_points_5"]),
        "hist_points_avg_last10": _deque_mean(state["recent_points_10"]),
        "hist_gf_avg_last5": _deque_mean(state["recent_gf_5"]),
        "hist_ga_avg_last5": _deque_mean(state["recent_ga_5"]),
        "hist_gd_avg_last5": _deque_mean(state["recent_gd_5"]),
        "hist_win_rate_last10": _deque_mean([1.0 if x == 1.0 else 0.0 for x in state["recent_results_10"]]),
        "hist_draw_rate_last10": _deque_mean([1.0 if x == 0.5 else 0.0 for x in state["recent_results_10"]]),
        "hist_loss_rate_last10": _deque_mean([1.0 if x == 0.0 else 0.0 for x in state["recent_results_10"]]),
        "hist_clean_sheet_rate_last5": _deque_mean(state["recent_clean_sheet_5"]),
        "hist_failed_to_score_rate_last5": _deque_mean(state["recent_failed_to_score_5"]),
        "hist_days_since_last_match": days_since,
        "hist_has_history": float(state["matches_played"] > 0),
    }


def snapshot_h2h_features(state: dict) -> dict:
    return {
        "h2h_matches_played_pre": float(state["h2h_matches_played"]),
        "h2h_points_a_avg_last3": _deque_mean(state["h2h_points_a_last3"]),
        "h2h_gd_a_avg_last3": _deque_mean(state["h2h_gd_a_last3"]),
        "h2h_total_goals_avg_last3": _deque_mean(state["h2h_total_goals_last3"]),
        "h2h_has_history": float(state["h2h_matches_played"] > 0),
    }


def update_states_from_score(
    state_a: dict,
    state_b: dict,
    h2h_state: dict,
    match_context: dict,
    goals_a: int,
    goals_b: int,
) -> tuple[dict, dict, dict]:
    goals_a = int(goals_a)
    goals_b = int(goals_b)
    goal_diff = goals_a - goals_b

    if goals_a > goals_b:
        points_a, points_b = 3, 0
        result_a, result_b = 1.0, 0.0
    elif goals_a == goals_b:
        points_a, points_b = 1, 1
        result_a, result_b = 0.5, 0.5
    else:
        points_a, points_b = 0, 3
        result_a, result_b = 0.0, 1.0

    tournament_w = get_tournament_weight(match_context.get("tournament", ""))
    home_a = float(match_context.get("home_a", 0))
    home_b = float(match_context.get("home_b", 0))
    elo_home_bonus = ELO_HOME_BONUS * home_a - ELO_HOME_BONUS * home_b
    gd_home_bonus = GD_HOME_BONUS * home_a - GD_HOME_BONUS * home_b

    exp_a = expected_elo_result(state_a["elo_overall"], state_b["elo_overall"], elo_home_bonus)
    exp_gd_a = (state_a["elo_goal_diff"] + gd_home_bonus - state_b["elo_goal_diff"]) / 100.0
    resid_gd_a = goal_diff - exp_gd_a

    state_a["elo_overall"] += K_ELO * tournament_w * (result_a - exp_a)
    state_b["elo_overall"] -= K_ELO * tournament_w * (result_a - exp_a)
    state_a["elo_goal_diff"] += K_GD * resid_gd_a
    state_b["elo_goal_diff"] -= K_GD * resid_gd_a

    for st, pts, gf, ga, gd, res in [
        (state_a, points_a, goals_a, goals_b, goal_diff, result_a),
        (state_b, points_b, goals_b, goals_a, -goal_diff, result_b),
    ]:
        st["matches_played"] += 1
        st["ewm_points"] = EWMA_ALPHA * pts + (1 - EWMA_ALPHA) * st["ewm_points"]
        st["ewm_goals_for"] = EWMA_ALPHA * gf + (1 - EWMA_ALPHA) * st["ewm_goals_for"]
        st["ewm_goals_against"] = EWMA_ALPHA * ga + (1 - EWMA_ALPHA) * st["ewm_goals_against"]
        st["ewm_goal_diff"] = EWMA_ALPHA * gd + (1 - EWMA_ALPHA) * st["ewm_goal_diff"]
        st["recent_points_5"].append(float(pts))
        st["recent_points_10"].append(float(pts))
        st["recent_gf_5"].append(float(gf))
        st["recent_ga_5"].append(float(ga))
        st["recent_gd_5"].append(float(gd))
        st["recent_results_10"].append(float(res))
        st["recent_clean_sheet_5"].append(float(ga == 0))
        st["recent_failed_to_score_5"].append(float(gf == 0))
        st["last_match_date"] = pd.Timestamp(match_context["date"])

    h2h_state["h2h_matches_played"] += 1
    h2h_state["h2h_points_a_last3"].append(float(points_a))
    h2h_state["h2h_gd_a_last3"].append(float(goal_diff))
    h2h_state["h2h_total_goals_last3"].append(float(goals_a + goals_b))

    return state_a, state_b, h2h_state


def build_history_feature_row(match_row: pd.Series, team_states: dict, h2h_states: dict) -> dict:
    gender = str(match_row["gender"])
    team_a = str(match_row["team_a"])
    team_b = str(match_row["team_b"])

    key_a = (gender, team_a)
    key_b = (gender, team_b)
    key_h2h = (gender, team_a, team_b)

    state_a = team_states.setdefault(key_a, init_team_state())
    state_b = team_states.setdefault(key_b, init_team_state())
    h2h_state = h2h_states.setdefault(key_h2h, init_h2h_state())

    snap_a = snapshot_team_features_full(state_a, match_row["date"])
    snap_b = snapshot_team_features_full(state_b, match_row["date"])
    snap_h2h = snapshot_h2h_features(h2h_state)

    row = {"match_id": match_row["match_id"]}
    for k, v in snap_a.items():
        row[f"{k}_a"] = v
    for k, v in snap_b.items():
        row[f"{k}_b"] = v

    delta_keys = [
        "hist_matches_played", "hist_elo_overall", "hist_elo_gd", "hist_ewm_points",
        "hist_ewm_gf", "hist_ewm_ga", "hist_ewm_gd", "hist_points_avg_last5",
        "hist_points_avg_last10", "hist_gf_avg_last5", "hist_ga_avg_last5",
        "hist_gd_avg_last5", "hist_win_rate_last10", "hist_draw_rate_last10",
        "hist_loss_rate_last10", "hist_clean_sheet_rate_last5",
        "hist_failed_to_score_rate_last5", "hist_days_since_last_match",
    ]
    for k in delta_keys:
        va = row.get(f"{k}_a", np.nan)
        vb = row.get(f"{k}_b", np.nan)
        row[f"{k}_diff"] = va - vb if pd.notna(va) and pd.notna(vb) else np.nan
        if "elo" in k:
            row[f"{k}_abs_diff"] = abs(row[f"{k}_diff"]) if pd.notna(row[f"{k}_diff"]) else np.nan

    row["hist_rest_days_diff"] = row.get("hist_days_since_last_match_a", np.nan) - row.get("hist_days_since_last_match_b", np.nan)
    row["hist_has_history_both"] = float(
        row.get("hist_has_history_a", 0) > 0 and row.get("hist_has_history_b", 0) > 0
    )

    row.update(snap_h2h)
    return row


def build_train_history_features_full(
    train_match_df: pd.DataFrame,
) -> tuple[pd.DataFrame, dict, dict]:
    temp = train_match_df.sort_values(["date", "match_id"]).reset_index(drop=True)
    team_states = {}
    h2h_states = {}
    rows = []

    for _, row in tqdm(temp.iterrows(), total=len(temp), desc="build_train_history_full"):
        hist_row = build_history_feature_row(row, team_states, h2h_states)
        rows.append(hist_row)

        gender = str(row["gender"])
        team_a = str(row["team_a"])
        team_b = str(row["team_b"])
        key_a = (gender, team_a)
        key_b = (gender, team_b)
        key_h2h = (gender, team_a, team_b)

        context = {
            "date": row["date"],
            "tournament": row["tournament"],
            "home_a": row.get("team_a_is_home", 0),
            "home_b": row.get("team_b_is_home", 0),
        }
        update_states_from_score(
            team_states[key_a],
            team_states[key_b],
            h2h_states[key_h2h],
            context,
            int(row["team_a_goals"]),
            int(row["team_b_goals"]),
        )

    hist_df = pd.DataFrame(rows)
    return hist_df, team_states, h2h_states


def simulate_freeze_feature_rows(
    future_match_df: pd.DataFrame,
    static_feature_df: pd.DataFrame,
    team_states_at_cutoff: dict,
    h2h_states_at_cutoff: dict,
) -> pd.DataFrame:
    """Bangun feature row pre-match untuk future matches secara freeze-safe.

    State performa tetap beku pada cutoff. Yang di-update hanya last_match_date agar
    days-since-last-match tetap masuk akal bila ada banyak future matches pada team yang sama.
    """
    team_states = copy.deepcopy(team_states_at_cutoff)
    h2h_states = copy.deepcopy(h2h_states_at_cutoff)
    static_lookup = static_feature_df.set_index("match_id")
    rows = []

    future_sorted = future_match_df.sort_values(["date", "match_id"]).reset_index(drop=True)
    for _, match_row in tqdm(future_sorted.iterrows(), total=len(future_sorted), desc="freeze_feature_rows"):
        hist_row = build_history_feature_row(match_row, team_states, h2h_states)
        static_row = static_lookup.loc[match_row["match_id"]].to_dict()
        feat_row = {**static_row, **hist_row}

        if "team_a_goals" in match_row.index:
            feat_row["actual_team_a_goals"] = match_row["team_a_goals"]
        if "team_b_goals" in match_row.index:
            feat_row["actual_team_b_goals"] = match_row["team_b_goals"]

        rows.append(feat_row)

        gender = str(match_row["gender"])
        key_a = (gender, str(match_row["team_a"]))
        key_b = (gender, str(match_row["team_b"]))
        team_states.setdefault(key_a, init_team_state())["last_match_date"] = pd.Timestamp(match_row["date"])
        team_states.setdefault(key_b, init_team_state())["last_match_date"] = pd.Timestamp(match_row["date"])

    return pd.DataFrame(rows)


train_history_full_df, final_team_states_after_train, final_h2h_states_after_train = build_train_history_features_full(train_match_base)

print(f"train_history_full_df shape : {train_history_full_df.shape}")
display(train_history_full_df.head(2))


build_train_history_full:   0%|          | 0/39386 [00:00<?, ?it/s]

train_history_full_df shape : (39386, 66)


,match_id,hist_matches_played_a,hist_elo_overall_a,hist_elo_gd_a,hist_ewm_points_a,hist_ewm_gf_a,hist_ewm_ga_a,hist_ewm_gd_a,hist_points_avg_last5_a,hist_points_avg_last10_a,hist_gf_avg_last5_a,hist_ga_avg_last5_a,hist_gd_avg_last5_a,hist_win_rate_last10_a,hist_draw_rate_last10_a,hist_loss_rate_last10_a,hist_clean_sheet_rate_last5_a,hist_failed_to_score_rate_last5_a,hist_days_since_last_match_a,hist_has_history_a,hist_matches_played_b,hist_elo_overall_b,hist_elo_gd_b,hist_ewm_points_b,hist_ewm_gf_b,hist_ewm_ga_b,hist_ewm_gd_b,hist_points_avg_last5_b,hist_points_avg_last10_b,hist_gf_avg_last5_b,hist_ga_avg_last5_b,hist_gd_avg_last5_b,hist_win_rate_last10_b,hist_draw_rate_last10_b,hist_loss_rate_last10_b,hist_clean_sheet_rate_last5_b,hist_failed_to_score_rate_last5_b,hist_days_since_last_match_b,hist_has_history_b,hist_matches_played_diff,hist_elo_overall_diff,hist_elo_overall_abs_diff,hist_elo_gd_diff,hist_elo_gd_abs_diff,hist_ewm_points_diff,hist_ewm_gf_diff,hist_ewm_ga_diff,hist_ewm_gd_diff,hist_points_avg_last5_diff,hist_points_avg_last10_diff,hist_gf_avg_last5_diff,hist_ga_avg_last5_diff,hist_gd_avg_last5_diff,hist_win_rate_last10_diff,hist_draw_rate_last10_diff,hist_loss_rate_last10_diff,hist_clean_sheet_rate_last5_diff,hist_failed_to_score_rate_last5_diff,hist_days_since_last_match_diff,hist_rest_days_diff,hist_has_history_both,h2h_matches_played_pre,h2h_points_a_avg_last3,h2h_gd_a_avg_last3,h2h_total_goals_avg_last3,h2h_has_history
0,M000001,0.0,1500.00000,0.0,1.0,1.20,1.20,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1500.00000,0.0,1.0,1.20,1.20,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0
1,M000002,1.0,1501.96989,0.6,1.0,0.78,0.78,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,98.0,1.0,1.0,1498.03011,-0.6,1.0,0.78,0.78,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,98.0,1.0,0.0,3.939779,3.939779,1.2,1.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0


### Catatan backbone

Target utama EXP04A bukan mengganti backbone.  
Karena itu, static block dan history-full block di atas dipertahankan agar eksperimen tetap fokus ke:

- objective refinement, dan
- tail-aware decisioning.


## 06. Temporal Holdout dan Scoreline Prior

Validasi tetap dibuat **fair dan leakage-safe** di level match, lalu state history dipotong pada cutoff train fold.

Scoreline prior dibangun dari train fold agar decoder punya baseline distribusi scoreline yang legal.


In [9]:
train_fold_base, valid_fold_base = make_time_based_holdout(train_match_base, valid_fraction=0.20)

train_static_fold = train_static_df.merge(train_fold_base[["match_id"]], on="match_id", how="inner")
valid_static_fold = train_static_df.merge(valid_fold_base[["match_id"]], on="match_id", how="inner")

train_fold_history_full, cutoff_team_states_valid, cutoff_h2h_states_valid = build_train_history_features_full(train_fold_base)

base_cols_to_drop_from_static = [c for c in train_match_base.columns if c != "match_id"]

train_feature_full = (
    train_fold_base
    .merge(train_static_fold.drop(columns=base_cols_to_drop_from_static, errors="ignore"), on="match_id", how="left")
    .merge(train_fold_history_full, on="match_id", how="left")
)

static_categorical_features = [
    "team_a", "team_b", "gender", "tournament", "venue_country",
    "team_a_confederation", "team_b_confederation", "pair_key", "confed_pair_key",
]

static_numeric_features = [
    "match_year", "match_month", "match_quarter", "match_dayofweek", "match_dayofyear",
    "match_is_weekend", "match_decade", "team_a_is_home", "team_b_is_home", "neutral",
    "home_side", "same_confederation", "is_friendly", "is_world_cup", "is_qualification",
    "is_nations_league", "tournament_weight_proxy", "altitude_venue", "temperature_venue",
    "population_a", "population_b", "population_diff", "population_abs_diff",
    "log_population_a", "log_population_b", "log_population_diff", "population_ratio_ab",
    "gdp_a", "gdp_b", "gdp_diff", "gdp_abs_diff", "log_gdp_a", "log_gdp_b",
    "log_gdp_diff", "gdp_ratio_ab", "distance_a", "distance_b", "distance_diff",
    "distance_abs_diff", "log_distance_a", "log_distance_b", "log_distance_diff",
    "distance_ratio_ab",
]

history_full_numeric_features = [
    "hist_matches_played_a", "hist_matches_played_b",
    "hist_elo_overall_a", "hist_elo_overall_b",
    "hist_elo_gd_a", "hist_elo_gd_b",
    "hist_ewm_points_a", "hist_ewm_points_b",
    "hist_ewm_gf_a", "hist_ewm_gf_b",
    "hist_ewm_ga_a", "hist_ewm_ga_b",
    "hist_ewm_gd_a", "hist_ewm_gd_b",
    "hist_points_avg_last5_a", "hist_points_avg_last5_b",
    "hist_points_avg_last10_a", "hist_points_avg_last10_b",
    "hist_gf_avg_last5_a", "hist_gf_avg_last5_b",
    "hist_ga_avg_last5_a", "hist_ga_avg_last5_b",
    "hist_gd_avg_last5_a", "hist_gd_avg_last5_b",
    "hist_win_rate_last10_a", "hist_win_rate_last10_b",
    "hist_draw_rate_last10_a", "hist_draw_rate_last10_b",
    "hist_loss_rate_last10_a", "hist_loss_rate_last10_b",
    "hist_clean_sheet_rate_last5_a", "hist_clean_sheet_rate_last5_b",
    "hist_failed_to_score_rate_last5_a", "hist_failed_to_score_rate_last5_b",
    "hist_days_since_last_match_a", "hist_days_since_last_match_b",
    "hist_has_history_a", "hist_has_history_b",
    "hist_matches_played_diff", "hist_elo_overall_diff", "hist_elo_overall_abs_diff",
    "hist_elo_gd_diff", "hist_elo_gd_abs_diff",
    "hist_ewm_points_diff", "hist_ewm_gf_diff", "hist_ewm_ga_diff", "hist_ewm_gd_diff",
    "hist_points_avg_last5_diff", "hist_points_avg_last10_diff",
    "hist_gf_avg_last5_diff", "hist_ga_avg_last5_diff", "hist_gd_avg_last5_diff",
    "hist_win_rate_last10_diff", "hist_draw_rate_last10_diff", "hist_loss_rate_last10_diff",
    "hist_clean_sheet_rate_last5_diff", "hist_failed_to_score_rate_last5_diff",
    "hist_rest_days_diff", "hist_has_history_both",
    "h2h_matches_played_pre", "h2h_points_a_avg_last3",
    "h2h_gd_a_avg_last3", "h2h_total_goals_avg_last3", "h2h_has_history",
]

full_feature_cols = static_categorical_features + static_numeric_features + history_full_numeric_features

feature_payload = {
    "static_categorical_features": static_categorical_features,
    "static_numeric_features": static_numeric_features,
    "history_full_numeric_features": history_full_numeric_features,
    "full_feature_cols": full_feature_cols,
}
with open(SUM_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_payload, f, indent=2)

def build_scoreline_prior(
    goal_a: pd.Series,
    goal_b: pd.Series,
    max_goals: int,
    alpha: float = 1.0,
) -> dict:
    counts = Counter()
    for a, b in zip(pd.to_numeric(goal_a, errors="coerce"), pd.to_numeric(goal_b, errors="coerce")):
        if pd.isna(a) or pd.isna(b):
            continue
        a_i = int(a)
        b_i = int(b)
        if a_i > max_goals or b_i > max_goals:
            a_i = min(a_i, max_goals)
            b_i = min(b_i, max_goals)
        counts[(a_i, b_i)] += 1

    total = 0.0
    scores = {}
    for a in range(max_goals + 1):
        for b in range(max_goals + 1):
            val = counts.get((a, b), 0) + alpha
            scores[(a, b)] = float(val)
            total += float(val)
    return {k: v / total for k, v in scores.items()}

DEFAULT_DECODER_GRID = {
    "MAX_GOALS": [7, 8, 10],
    "w_direct": [0.5, 1.0],
    "w_total": [1.0, 1.5],
    "w_gd": [1.0, 1.5],
    "w_outcome": [1.5, 2.0],
    "w_prior": [0.2, 0.5],
}
scoreline_prior_lookup = {
    mg: build_scoreline_prior(train_fold_base["team_a_goals"], train_fold_base["team_b_goals"], max_goals=int(mg), alpha=1.0)
    for mg in DEFAULT_DECODER_GRID["MAX_GOALS"]
}

print(f"train_fold_base shape    : {train_fold_base.shape}")
print(f"valid_fold_base shape    : {valid_fold_base.shape}")
print(f"train_feature_full shape : {train_feature_full.shape}")
print(f"#full_feature_cols       : {len(full_feature_cols)}")
display(train_feature_full.head(2))


build_train_history_full:   0%|          | 0/31508 [00:00<?, ?it/s]

train_fold_base shape    : (31508, 22)
valid_fold_base shape    : (7878, 22)
train_feature_full shape : (31508, 127)
#full_feature_cols       : 116


,match_id,date,gender,tournament,venue_country,neutral,altitude_venue,temperature_venue,team_a,team_b,team_a_is_home,team_b_is_home,team_a_confederation,team_b_confederation,team_a_population,team_b_population,team_a_gdp_per_capita,team_b_gdp_per_capita,team_a_distance_travel,team_b_distance_travel,team_a_goals,team_b_goals,match_year,match_month,match_quarter,match_dayofweek,match_dayofyear,match_is_weekend,match_decade,home_side,same_confederation,is_friendly,is_world_cup,is_qualification,is_nations_league,tournament_weight_proxy,population_a,population_b,population_diff,population_abs_diff,log_population_a,log_population_b,log_population_diff,population_ratio_ab,gdp_a,gdp_b,gdp_diff,gdp_abs_diff,log_gdp_a,log_gdp_b,log_gdp_diff,gdp_ratio_ab,distance_a,distance_b,distance_diff,distance_abs_diff,log_distance_a,log_distance_b,log_distance_diff,distance_ratio_ab,pair_key,confed_pair_key,hist_matches_played_a,hist_elo_overall_a,hist_elo_gd_a,hist_ewm_points_a,hist_ewm_gf_a,hist_ewm_ga_a,hist_ewm_gd_a,hist_points_avg_last5_a,hist_points_avg_last10_a,hist_gf_avg_last5_a,hist_ga_avg_last5_a,hist_gd_avg_last5_a,hist_win_rate_last10_a,hist_draw_rate_last10_a,hist_loss_rate_last10_a,hist_clean_sheet_rate_last5_a,hist_failed_to_score_rate_last5_a,hist_days_since_last_match_a,hist_has_history_a,hist_matches_played_b,hist_elo_overall_b,hist_elo_gd_b,hist_ewm_points_b,hist_ewm_gf_b,hist_ewm_ga_b,hist_ewm_gd_b,hist_points_avg_last5_b,hist_points_avg_last10_b,hist_gf_avg_last5_b,hist_ga_avg_last5_b,hist_gd_avg_last5_b,hist_win_rate_last10_b,hist_draw_rate_last10_b,hist_loss_rate_last10_b,hist_clean_sheet_rate_last5_b,hist_failed_to_score_rate_last5_b,hist_days_since_last_match_b,hist_has_history_b,hist_matches_played_diff,hist_elo_overall_diff,hist_elo_overall_abs_diff,hist_elo_gd_diff,hist_elo_gd_abs_diff,hist_ewm_points_diff,hist_ewm_gf_diff,hist_ewm_ga_diff,hist_ewm_gd_diff,hist_points_avg_last5_diff,hist_points_avg_last10_diff,hist_gf_avg_last5_diff,hist_ga_avg_last5_diff,hist_gd_avg_last5_diff,hist_win_rate_last10_diff,hist_draw_rate_last10_diff,hist_loss_rate_last10_diff,hist_clean_sheet_rate_last5_diff,hist_failed_to_score_rate_last5_diff,hist_days_since_last_match_diff,hist_rest_days_diff,hist_has_history_both,h2h_matches_played_pre,h2h_points_a_avg_last3,h2h_gd_a_avg_last3,h2h_total_goals_avg_last3,h2h_has_history
0,M000001,1872-11-30,M,Friendly,Scotland,0,NaN,NaN,England,Scotland,0.0,1.0,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1872,11,4,5,335,1.0,1870,-1.0,1.0,1.0,0.0,0.0,0.0,0.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,England__VS__Scotland,UEFA__VS__UEFA,0.0,1500.00000,0.0,1.0,1.20,1.20,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1500.00000,0.0,1.0,1.20,1.20,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0
1,M000002,1873-03-08,M,Friendly,England,0,NaN,NaN,England,Scotland,1.0,0.0,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,4.0,2.0,1873,3,1,5,67,1.0,1870,1.0,1.0,1.0,0.0,0.0,0.0,0.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,England__VS__Scotland,UEFA__VS__UEFA,1.0,1501.96989,0.6,1.0,0.78,0.78,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,98.0,1.0,1.0,1498.03011,-0.6,1.0,0.78,0.78,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,98.0,1.0,0.0,3.939779,3.939779,1.2,1.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0


## 07. Target Construction

Target utama tetap:

- `y_goal_a`
- `y_goal_b`
- `y_total`
- `y_gd`
- `y_outcome`

Selain itu kita tambahkan target tail / blowout.

Target-target tambahan ini **tidak menggantikan target utama skor**, tetapi membantu model mengenali bahwa sebagian pertandingan berada di rezim yang tidak normal / tidak moderat.

Dengan kata lain, model tidak cuma belajar **“berapa gol?”**, tapi juga **“apakah ini pertandingan yang berpotensi meledak?”**


In [10]:
def build_outcome_target(goal_a: pd.Series, goal_b: pd.Series) -> pd.Series:
    goal_a = pd.to_numeric(goal_a, errors="coerce")
    goal_b = pd.to_numeric(goal_b, errors="coerce")
    return pd.Series(np.where(goal_a > goal_b, 0, np.where(goal_a == goal_b, 1, 2)), index=goal_a.index)

train_feature_full["y_goal_a"] = train_feature_full["team_a_goals"].astype(float)
train_feature_full["y_goal_b"] = train_feature_full["team_b_goals"].astype(float)
train_feature_full["y_total"] = train_feature_full["team_a_goals"].astype(float) + train_feature_full["team_b_goals"].astype(float)
train_feature_full["y_gd"] = train_feature_full["team_a_goals"].astype(float) - train_feature_full["team_b_goals"].astype(float)
train_feature_full["y_outcome"] = build_outcome_target(train_feature_full["team_a_goals"], train_feature_full["team_b_goals"]).astype(int)

train_feature_full["is_big_margin_5plus"] = (train_feature_full["y_gd"].abs() >= 5).astype(int)
train_feature_full["is_big_margin_7plus"] = (train_feature_full["y_gd"].abs() >= 7).astype(int)
train_feature_full["is_high_total_6plus"] = (train_feature_full["y_total"] >= 6).astype(int)
train_feature_full["is_high_total_8plus"] = (train_feature_full["y_total"] >= 8).astype(int)
train_feature_full["is_team_a_blowout"] = (train_feature_full["y_gd"] >= 5).astype(int)
train_feature_full["is_team_b_blowout"] = (train_feature_full["y_gd"] <= -5).astype(int)

tail_target_definitions = {
    "is_big_margin_5plus": "1 jika abs(goal_diff) >= 5",
    "is_big_margin_7plus": "1 jika abs(goal_diff) >= 7",
    "is_high_total_6plus": "1 jika total_goals >= 6",
    "is_high_total_8plus": "1 jika total_goals >= 8",
    "is_team_a_blowout": "1 jika goal_diff >= 5",
    "is_team_b_blowout": "1 jika goal_diff <= -5",
}
with open(SUM_DIR / "tail_target_definitions.json", "w", encoding="utf-8") as f:
    json.dump(tail_target_definitions, f, indent=2)

display(train_feature_full[
    ["match_id", "y_goal_a", "y_goal_b", "y_total", "y_gd", "y_outcome",
     "is_big_margin_5plus", "is_big_margin_7plus", "is_high_total_6plus", "is_high_total_8plus"]
].head())

print("[OK] tail_target_definitions.json disimpan.")


,match_id,y_goal_a,y_goal_b,y_total,y_gd,y_outcome,is_big_margin_5plus,is_big_margin_7plus,is_high_total_6plus,is_high_total_8plus
0,M000001,0.0,0.0,0.0,0.0,1,0,0,0,0
1,M000002,4.0,2.0,6.0,2.0,0,0,0,1,0
2,M000003,1.0,2.0,3.0,-1.0,2,0,0,0,0
3,M000004,2.0,2.0,4.0,0.0,1,0,0,0,0
4,M000005,0.0,3.0,3.0,-3.0,2,0,0,0,0


[OK] tail_target_definitions.json disimpan.


## 08. Anchor RMSE Baseline

V0 ini adalah baseline internal EXP04A.  
Kita mempertahankan backbone history-full yang sama, lalu memakai objective deterministic lama sebagai anchor:

- `goal_a` = RMSE
- `goal_b` = RMSE
- `total` = RMSE
- `gd` = RMSE
- `outcome` = MultiClass

Setelah model selesai, decoder dituning secara **raw-output-first**.


In [11]:
goal_reg_params_rmse = dict(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=1500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=200,
    od_type="Iter",
    od_wait=200,
)

goal_reg_params_poisson = dict(
    loss_function="Poisson",
    eval_metric="Poisson",
    iterations=1500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=200,
    od_type="Iter",
    od_wait=200,
)

TWEEDIE_VARIANCE_POWER = 1.3
goal_reg_params_tweedie = dict(
    loss_function=f"Tweedie:variance_power={TWEEDIE_VARIANCE_POWER}",
    eval_metric=f"Tweedie:variance_power={TWEEDIE_VARIANCE_POWER}",
    iterations=1500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=200,
    od_type="Iter",
    od_wait=200,
)

outcome_clf_params = dict(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=1500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=200,
    od_type="Iter",
    od_wait=200,
)

tail_clf_params = dict(
    loss_function="Logloss",
    eval_metric="Logloss",
    iterations=1200,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=200,
    od_type="Iter",
    od_wait=150,
)

objective_configurations = {
    "anchor_rmse": {
        "goal_a": goal_reg_params_rmse,
        "goal_b": goal_reg_params_rmse,
        "total": goal_reg_params_rmse,
        "gd": goal_reg_params_rmse,
        "outcome": outcome_clf_params,
    },
    "poisson": {
        "goal_a": goal_reg_params_poisson,
        "goal_b": goal_reg_params_poisson,
        "total": goal_reg_params_rmse,
        "gd": goal_reg_params_rmse,
        "outcome": outcome_clf_params,
    },
    "tweedie": {
        "goal_a": goal_reg_params_tweedie,
        "goal_b": goal_reg_params_tweedie,
        "total": goal_reg_params_rmse,
        "gd": goal_reg_params_rmse,
        "outcome": outcome_clf_params,
        "notes": {
            "variance_power": TWEEDIE_VARIANCE_POWER,
            "reason": "safe default, masih dekat ke count-like target tanpa grid brutal",
        },
    },
    "tail_classifier": tail_clf_params,
}
with open(SUM_DIR / "objective_configurations.json", "w", encoding="utf-8") as f:
    json.dump(objective_configurations, f, indent=2)

def _prepare_feature_df(df: pd.DataFrame, feature_cols, categorical_cols):
    X = df[feature_cols].copy()
    for col in categorical_cols:
        if col in X.columns:
            X[col] = X[col].fillna("MISSING").astype(str)
    num_cols = [c for c in X.columns if c not in categorical_cols]
    for col in num_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X[col] = X[col].replace([np.inf, -np.inf], np.nan)
    return X


def _make_inner_temporal_split(df: pd.DataFrame, eval_fraction: float = 0.15):
    temp = df.sort_values(["date", "match_id"]).reset_index(drop=True)
    n_eval = max(1, int(math.ceil(len(temp) * eval_fraction)))
    n_eval = min(n_eval, max(1, len(temp) - 1))
    split_idx = len(temp) - n_eval
    inner_train = temp.iloc[:split_idx].copy()
    inner_eval = temp.iloc[split_idx:].copy()
    return inner_train, inner_eval


def _get_cat_indices(df: pd.DataFrame, cat_cols: list[str]) -> list[int]:
    return [df.columns.get_loc(c) for c in cat_cols if c in df.columns]


def _fit_regressor(train_df, feature_cols, cat_cols, target_col, params):
    inner_train, inner_eval = _make_inner_temporal_split(train_df, eval_fraction=0.15)

    X_tr = _prepare_feature_df(inner_train, feature_cols, cat_cols)
    X_ev = _prepare_feature_df(inner_eval, feature_cols, cat_cols)

    train_pool = Pool(X_tr, inner_train[target_col].astype(float).values, cat_features=_get_cat_indices(X_tr, cat_cols))
    eval_pool = Pool(X_ev, inner_eval[target_col].astype(float).values, cat_features=_get_cat_indices(X_ev, cat_cols))

    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=eval_pool, use_best_model=True)
    return model


def _fit_classifier(train_df, feature_cols, cat_cols, target_series, params):
    inner_train, inner_eval = _make_inner_temporal_split(train_df, eval_fraction=0.15)

    y_tr = target_series.loc[inner_train.index].values
    y_ev = target_series.loc[inner_eval.index].values

    X_tr = _prepare_feature_df(inner_train, feature_cols, cat_cols)
    X_ev = _prepare_feature_df(inner_eval, feature_cols, cat_cols)

    train_pool = Pool(X_tr, y_tr, cat_features=_get_cat_indices(X_tr, cat_cols))
    eval_pool = Pool(X_ev, y_ev, cat_features=_get_cat_indices(X_ev, cat_cols))

    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=eval_pool, use_best_model=True)
    return model


def train_variant_bundle(
    train_df: pd.DataFrame,
    feature_cols: list[str],
    cat_cols: list[str],
    goal_params: dict,
    total_params: dict,
    gd_params: dict,
    outcome_params: dict,
    variant_name: str,
) -> dict:
    bundle = {
        "variant_name": variant_name,
        "feature_cols": feature_cols,
        "cat_cols": cat_cols,
        "goal_a": _fit_regressor(train_df, feature_cols, cat_cols, "y_goal_a", goal_params),
        "goal_b": _fit_regressor(train_df, feature_cols, cat_cols, "y_goal_b", goal_params),
        "total": _fit_regressor(train_df, feature_cols, cat_cols, "y_total", total_params),
        "gd": _fit_regressor(train_df, feature_cols, cat_cols, "y_gd", gd_params),
        "outcome": _fit_classifier(train_df, feature_cols, cat_cols, train_df["y_outcome"], outcome_params),
    }
    return bundle


def predict_raw_outputs_for_df(
    feature_df: pd.DataFrame,
    trained_models: dict,
) -> pd.DataFrame:
    X = _prepare_feature_df(feature_df, trained_models["feature_cols"], trained_models["cat_cols"])
    pred_goal_a = trained_models["goal_a"].predict(X)
    pred_goal_b = trained_models["goal_b"].predict(X)
    pred_total = trained_models["total"].predict(X)
    pred_gd = trained_models["gd"].predict(X)
    pred_outcome_proba = trained_models["outcome"].predict_proba(X)
    outcome_classes = list(trained_models["outcome"].classes_)

    out = feature_df[["match_id", "tournament"]].copy()
    if "actual_team_a_goals" in feature_df.columns:
        out["actual_team_a_goals"] = feature_df["actual_team_a_goals"].values
    if "actual_team_b_goals" in feature_df.columns:
        out["actual_team_b_goals"] = feature_df["actual_team_b_goals"].values

    out["pred_goal_a_cont"] = pred_goal_a
    out["pred_goal_b_cont"] = pred_goal_b
    out["pred_total_cont"] = pred_total
    out["pred_gd_cont"] = pred_gd

    for cls in [0, 1, 2]:
        if cls in outcome_classes:
            out[f"pred_outcome_proba_{cls}"] = pred_outcome_proba[:, outcome_classes.index(cls)]
        else:
            out[f"pred_outcome_proba_{cls}"] = 0.0
    return out


def decode_single_match_score(
    pred_goal_a: float,
    pred_goal_b: float,
    pred_total: float,
    pred_gd: float,
    pred_outcome_proba: np.ndarray,
    scoreline_prior: dict,
    max_goals: int,
    w_direct: float,
    w_total: float,
    w_gd: float,
    w_outcome: float,
    w_prior: float,
    eps: float = 1e-12,
) -> tuple[int, int]:
    best_score = None
    best_pair = (0, 0)

    for a in range(max_goals + 1):
        for b in range(max_goals + 1):
            cand_outcome = 0 if a > b else (1 if a == b else 2)
            score = (
                w_direct * (abs(a - pred_goal_a) + abs(b - pred_goal_b))
                + w_total * abs((a + b) - pred_total)
                + w_gd * abs((a - b) - pred_gd)
                + w_outcome * (-math.log(float(pred_outcome_proba[cand_outcome]) + eps))
                + w_prior * (-math.log(float(scoreline_prior.get((a, b), eps)) + eps))
            )
            if best_score is None or score < best_score:
                best_score = score
                best_pair = (a, b)
    return best_pair


def tune_decoder_from_raw_outputs(
    raw_df: pd.DataFrame,
    decoder_grid: dict,
    scoreline_prior: dict,
) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    rows = []
    all_params = list(product(
        decoder_grid["MAX_GOALS"],
        decoder_grid["w_direct"],
        decoder_grid["w_total"],
        decoder_grid["w_gd"],
        decoder_grid["w_outcome"],
        decoder_grid["w_prior"],
    ))

    for mg, wd, wt, wgd, wo, wp in tqdm(all_params, desc="tune_decoder"):
        prior_for_mg = scoreline_prior[int(mg)] if int(mg) in scoreline_prior else scoreline_prior

        preds_a, preds_b = [], []
        for _, row in raw_df.iterrows():
            pair = decode_single_match_score(
                pred_goal_a=float(row["pred_goal_a_cont"]),
                pred_goal_b=float(row["pred_goal_b_cont"]),
                pred_total=float(row["pred_total_cont"]),
                pred_gd=float(row["pred_gd_cont"]),
                pred_outcome_proba=np.array(
                    [row["pred_outcome_proba_0"], row["pred_outcome_proba_1"], row["pred_outcome_proba_2"]],
                    dtype=float,
                ),
                scoreline_prior=prior_for_mg,
                max_goals=int(mg),
                w_direct=float(wd),
                w_total=float(wt),
                w_gd=float(wgd),
                w_outcome=float(wo),
                w_prior=float(wp),
            )
            preds_a.append(pair[0])
            preds_b.append(pair[1])

        score = awmae_score(
            raw_df["actual_team_a_goals"],
            raw_df["actual_team_b_goals"],
            preds_a,
            preds_b,
            raw_df["tournament"],
        )
        rows.append({
            "MAX_GOALS": int(mg),
            "w_direct": float(wd),
            "w_total": float(wt),
            "w_gd": float(wgd),
            "w_outcome": float(wo),
            "w_prior": float(wp),
            "valid_awmae": float(score),
        })

    grid_df = pd.DataFrame(rows).sort_values("valid_awmae").reset_index(drop=True)
    best = grid_df.iloc[0].to_dict()
    best_prior = scoreline_prior[int(best["MAX_GOALS"])] if int(best["MAX_GOALS"]) in scoreline_prior else scoreline_prior

    pred_rows = []
    for _, row in raw_df.iterrows():
        pair = decode_single_match_score(
            pred_goal_a=float(row["pred_goal_a_cont"]),
            pred_goal_b=float(row["pred_goal_b_cont"]),
            pred_total=float(row["pred_total_cont"]),
            pred_gd=float(row["pred_gd_cont"]),
            pred_outcome_proba=np.array(
                [row["pred_outcome_proba_0"], row["pred_outcome_proba_1"], row["pred_outcome_proba_2"]],
                dtype=float,
            ),
            scoreline_prior=best_prior,
            max_goals=int(best["MAX_GOALS"]),
            w_direct=float(best["w_direct"]),
            w_total=float(best["w_total"]),
            w_gd=float(best["w_gd"]),
            w_outcome=float(best["w_outcome"]),
            w_prior=float(best["w_prior"]),
        )
        pred_rows.append({
            "match_id": row["match_id"],
            "pred_team_a_goals": int(pair[0]),
            "pred_team_b_goals": int(pair[1]),
        })

    pred_df = pd.DataFrame(pred_rows)
    return grid_df, best, pred_df


def train_tail_models(
    train_df: pd.DataFrame,
    feature_cols: list[str],
    cat_cols: list[str],
) -> dict:
    tail_models = {
        "clf_big_margin_5plus": _fit_classifier(train_df, feature_cols, cat_cols, train_df["is_big_margin_5plus"], tail_clf_params),
        "clf_big_margin_7plus": _fit_classifier(train_df, feature_cols, cat_cols, train_df["is_big_margin_7plus"], tail_clf_params),
        "clf_high_total_6plus": _fit_classifier(train_df, feature_cols, cat_cols, train_df["is_high_total_6plus"], tail_clf_params),
        "clf_high_total_8plus": _fit_classifier(train_df, feature_cols, cat_cols, train_df["is_high_total_8plus"], tail_clf_params),
        "feature_cols": feature_cols,
        "cat_cols": cat_cols,
    }
    return tail_models


def predict_tail_probabilities_for_df(
    feature_df: pd.DataFrame,
    tail_models: dict,
) -> pd.DataFrame:
    X = _prepare_feature_df(feature_df, tail_models["feature_cols"], tail_models["cat_cols"])
    out = feature_df[["match_id"]].copy()

    mapping = {
        "clf_big_margin_5plus": "prob_big_margin_5plus",
        "clf_big_margin_7plus": "prob_big_margin_7plus",
        "clf_high_total_6plus": "prob_high_total_6plus",
        "clf_high_total_8plus": "prob_high_total_8plus",
    }

    for model_key, out_col in mapping.items():
        model = tail_models[model_key]
        probs = model.predict_proba(X)
        classes = list(model.classes_)
        pos_idx = classes.index(1) if 1 in classes else -1
        out[out_col] = probs[:, pos_idx] if pos_idx >= 0 else 0.0

    return out


def _tail_penalty_for_candidate(
    a: int,
    b: int,
    tail_row: pd.Series,
    w_bm5: float,
    w_bm7: float,
    w_ht: float,
    eps: float = 1e-12,
) -> float:
    p_bm5 = float(np.clip(tail_row.get("prob_big_margin_5plus", 0.0), eps, 1 - eps))
    p_bm7 = float(np.clip(tail_row.get("prob_big_margin_7plus", 0.0), eps, 1 - eps))
    p_ht = float(np.clip(max(
        tail_row.get("prob_high_total_6plus", 0.0),
        tail_row.get("prob_high_total_8plus", 0.0),
    ), eps, 1 - eps))

    cost = 0.0

    if abs(a - b) >= 5:
        cost += w_bm5 * (-math.log(p_bm5))
    else:
        cost += 0.25 * w_bm5 * (-math.log(1.0 - p_bm5))

    if abs(a - b) >= 7:
        cost += w_bm7 * (-math.log(p_bm7))
    else:
        cost += 0.25 * w_bm7 * (-math.log(1.0 - p_bm7))

    if (a + b) >= 6 or max(a, b) >= 6:
        cost += w_ht * (-math.log(p_ht))
    else:
        cost += 0.25 * w_ht * (-math.log(1.0 - p_ht))

    return float(cost)


def decode_single_match_tail_aware(
    raw_row: pd.Series,
    tail_row: pd.Series,
    decoder_params: dict,
    scoreline_prior: dict,
    eps: float = 1e-12,
) -> tuple[int, int]:
    max_goals = int(decoder_params["MAX_GOALS"])
    best_score = None
    best_pair = (0, 0)

    for a in range(max_goals + 1):
        for b in range(max_goals + 1):
            cand_outcome = 0 if a > b else (1 if a == b else 2)

            anchor_cost = (
                float(decoder_params["w_direct"]) * (abs(a - float(raw_row["pred_goal_a_cont"])) + abs(b - float(raw_row["pred_goal_b_cont"])))
                + float(decoder_params["w_total"]) * abs((a + b) - float(raw_row["pred_total_cont"]))
                + float(decoder_params["w_gd"]) * abs((a - b) - float(raw_row["pred_gd_cont"]))
                + float(decoder_params["w_outcome"]) * (-math.log(float(raw_row[f"pred_outcome_proba_{cand_outcome}"]) + eps))
                + float(decoder_params["w_prior"]) * (-math.log(float(scoreline_prior.get((a, b), eps)) + eps))
            )

            tail_cost = _tail_penalty_for_candidate(
                a=a,
                b=b,
                tail_row=tail_row,
                w_bm5=float(decoder_params["w_bm5"]),
                w_bm7=float(decoder_params["w_bm7"]),
                w_ht=float(decoder_params["w_ht"]),
                eps=eps,
            )
            score = anchor_cost + tail_cost

            if best_score is None or score < best_score:
                best_score = score
                best_pair = (a, b)

    return best_pair


def tune_tail_aware_decoder_from_raw_outputs(
    raw_df: pd.DataFrame,
    tail_prob_df: pd.DataFrame,
    decoder_grid: dict,
    scoreline_prior: dict,
) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    merged = raw_df.merge(tail_prob_df, on="match_id", how="left")
    rows = []

    keys = ["MAX_GOALS", "w_direct", "w_total", "w_gd", "w_outcome", "w_prior", "w_bm5", "w_bm7", "w_ht"]
    value_lists = [decoder_grid[k] for k in keys]

    for combo in tqdm(list(product(*value_lists)), desc="tune_tail_decoder"):
        params = dict(zip(keys, combo))
        prior_for_mg = scoreline_prior[int(params["MAX_GOALS"])] if int(params["MAX_GOALS"]) in scoreline_prior else scoreline_prior

        preds_a, preds_b = [], []
        for _, row in merged.iterrows():
            pair = decode_single_match_tail_aware(
                raw_row=row,
                tail_row=row,
                decoder_params=params,
                scoreline_prior=prior_for_mg,
            )
            preds_a.append(pair[0])
            preds_b.append(pair[1])

        score = awmae_score(
            merged["actual_team_a_goals"],
            merged["actual_team_b_goals"],
            preds_a,
            preds_b,
            merged["tournament"],
        )
        rows.append({**{k: float(v) if k != "MAX_GOALS" else int(v) for k, v in params.items()}, "valid_awmae": float(score)})

    grid_df = pd.DataFrame(rows).sort_values("valid_awmae").reset_index(drop=True)
    best = grid_df.iloc[0].to_dict()
    best_prior = scoreline_prior[int(best["MAX_GOALS"])] if int(best["MAX_GOALS"]) in scoreline_prior else scoreline_prior

    pred_rows = []
    for _, row in merged.iterrows():
        pair = decode_single_match_tail_aware(
            raw_row=row,
            tail_row=row,
            decoder_params=best,
            scoreline_prior=best_prior,
        )
        pred_rows.append({
            "match_id": row["match_id"],
            "pred_team_a_goals": int(pair[0]),
            "pred_team_b_goals": int(pair[1]),
        })

    pred_df = pd.DataFrame(pred_rows)
    return grid_df, best, pred_df


valid_freeze_feature_full = simulate_freeze_feature_rows(
    valid_fold_base,
    valid_static_fold,
    cutoff_team_states_valid,
    cutoff_h2h_states_valid,
)
print(f"valid_freeze_feature_full shape : {valid_freeze_feature_full.shape}")
display(valid_freeze_feature_full.head(2))


freeze_feature_rows:   0%|          | 0/7878 [00:00<?, ?it/s]

valid_freeze_feature_full shape : (7878, 129)


,date,gender,tournament,venue_country,neutral,altitude_venue,temperature_venue,team_a,team_b,team_a_is_home,team_b_is_home,team_a_confederation,team_b_confederation,team_a_population,team_b_population,team_a_gdp_per_capita,team_b_gdp_per_capita,team_a_distance_travel,team_b_distance_travel,team_a_goals,team_b_goals,match_year,match_month,match_quarter,match_dayofweek,match_dayofyear,match_is_weekend,match_decade,home_side,same_confederation,is_friendly,is_world_cup,is_qualification,is_nations_league,tournament_weight_proxy,population_a,population_b,population_diff,population_abs_diff,log_population_a,log_population_b,log_population_diff,population_ratio_ab,gdp_a,gdp_b,gdp_diff,gdp_abs_diff,log_gdp_a,log_gdp_b,log_gdp_diff,gdp_ratio_ab,distance_a,distance_b,distance_diff,distance_abs_diff,log_distance_a,log_distance_b,log_distance_diff,distance_ratio_ab,pair_key,confed_pair_key,match_id,hist_matches_played_a,hist_elo_overall_a,hist_elo_gd_a,hist_ewm_points_a,hist_ewm_gf_a,hist_ewm_ga_a,hist_ewm_gd_a,hist_points_avg_last5_a,hist_points_avg_last10_a,hist_gf_avg_last5_a,hist_ga_avg_last5_a,hist_gd_avg_last5_a,hist_win_rate_last10_a,hist_draw_rate_last10_a,hist_loss_rate_last10_a,hist_clean_sheet_rate_last5_a,hist_failed_to_score_rate_last5_a,hist_days_since_last_match_a,hist_has_history_a,hist_matches_played_b,hist_elo_overall_b,hist_elo_gd_b,hist_ewm_points_b,hist_ewm_gf_b,hist_ewm_ga_b,hist_ewm_gd_b,hist_points_avg_last5_b,hist_points_avg_last10_b,hist_gf_avg_last5_b,hist_ga_avg_last5_b,hist_gd_avg_last5_b,hist_win_rate_last10_b,hist_draw_rate_last10_b,hist_loss_rate_last10_b,hist_clean_sheet_rate_last5_b,hist_failed_to_score_rate_last5_b,hist_days_since_last_match_b,hist_has_history_b,hist_matches_played_diff,hist_elo_overall_diff,hist_elo_overall_abs_diff,hist_elo_gd_diff,hist_elo_gd_abs_diff,hist_ewm_points_diff,hist_ewm_gf_diff,hist_ewm_ga_diff,hist_ewm_gd_diff,hist_points_avg_last5_diff,hist_points_avg_last10_diff,hist_gf_avg_last5_diff,hist_ga_avg_last5_diff,hist_gd_avg_last5_diff,hist_win_rate_last10_diff,hist_draw_rate_last10_diff,hist_loss_rate_last10_diff,hist_clean_sheet_rate_last5_diff,hist_failed_to_score_rate_last5_diff,hist_days_since_last_match_diff,hist_rest_days_diff,hist_has_history_both,h2h_matches_played_pre,h2h_points_a_avg_last3,h2h_gd_a_avg_last3,h2h_total_goals_avg_last3,h2h_has_history,actual_team_a_goals,actual_team_b_goals
0,2005-02-01,M,Friendly,Trinidad and Tobago,0,NaN,26.137575,Haiti,Trinidad and Tobago,0.0,1.0,CONCACAF,CONCACAF,8360225.0,1332203.0,781.276658,12327.280713,NaN,NaN,0.0,1.0,2005,2,1,1,32,0.0,2000,-1.0,1.0,1.0,0.0,0.0,0.0,0.96,8360225.0,1332203.0,7028022.0,7028022.0,15.938996,14.102345,1.836651,6.275489,781.276658,12327.280713,-11546.004055,11546.004055,6.662208,9.419651,-2.757443,0.063378,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Haiti__VS__Trinidad and Tobago,CONCACAF__VS__CONCACAF,M028954,316.0,1569.339685,82.477954,1.197707,1.590038,1.409747,0.180291,1.6,1.4,1.4,1.0,0.4,0.4,0.2,0.4,0.4,0.2,16.0,1.0,486.0,1608.814328,78.210671,2.520151,1.755770,0.350161,1.405609,2.4,2.4,2.0,0.4,1.6,0.8,0.0,0.2,0.8,0.0,9.0,1.0,-170.0,-39.474643,39.474643,4.267283,4.267283,-1.322445,-0.165732,1.059586,-1.225318,-0.8,-1.0,-0.6,0.6,-1.2,-0.4,0.2,0.2,-0.4,0.2,7.0,7.0,1.0,26.0,1.333333,-0.333333,2.333333,1.0,0.0,1.0
1,2005-02-01,W,Four Nations Tournament,China PR,1,NaN,NaN,Australia,Russia,0.0,1.0,AFC,UEFA,19017963.0,146844839.0,34080.999895,NaN,NaN,NaN,5.0,0.0,2005,2,1,1,32,0.0,2000,-1.0,0.0,0.0,0.0,0.0,0.0,1.20,19017963.0,146844839.0,-127826876.0,127826876.0,16.760895,18.804887,-2.043993,0.129511,34080.999895,NaN,NaN,NaN,10.436525,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Australia__VS__Russia,AFC__VS__UEFA,W002556,95.0,1689.515298,461.015218,1.077697,0.813742,1.569004,-0.755262,1.4,1.4,0.8,1.2,-0.4,0.4,0.2,0.4,0.4,0.2,2.0,1.0,76.0,1688.988335,293.337716,0.422342,0.921752,1.777675,-0.855924,0.6,1.0,1.2,1.6,-0.4,0.3,0.1,0.6,0.2,0.4,2.0,1.0,19.0,0.526963,0.526963,167.677502,167.677502,0.655356,-0.108010,-0.20867

In [12]:
anchor_bundle = train_variant_bundle(
    train_df=train_feature_full,
    feature_cols=full_feature_cols,
    cat_cols=static_categorical_features,
    goal_params=goal_reg_params_rmse,
    total_params=goal_reg_params_rmse,
    gd_params=goal_reg_params_rmse,
    outcome_params=outcome_clf_params,
    variant_name="anchor_rmse",
)

raw_anchor_valid = predict_raw_outputs_for_df(valid_freeze_feature_full, anchor_bundle)
decoder_grid_anchor, best_decoder_anchor, pred_anchor_valid = tune_decoder_from_raw_outputs(
    raw_df=raw_anchor_valid,
    decoder_grid=DEFAULT_DECODER_GRID,
    scoreline_prior=scoreline_prior_lookup,
)
valid_pred_match_anchor = raw_anchor_valid.merge(pred_anchor_valid, on="match_id", how="left")
valid_pred_match_anchor.to_csv(PRED_DIR / "valid_pred_match_anchor.csv", index=False)
decoder_grid_anchor.to_csv(SUM_DIR / "decoder_grid_anchor.csv", index=False)

awmae_anchor = awmae_score(
    valid_pred_match_anchor["actual_team_a_goals"],
    valid_pred_match_anchor["actual_team_b_goals"],
    valid_pred_match_anchor["pred_team_a_goals"],
    valid_pred_match_anchor["pred_team_b_goals"],
    valid_pred_match_anchor["tournament"],
)

print(f"[RESULT] anchor_rmse valid AW-MAE = {awmae_anchor:.6f}")
display(valid_pred_match_anchor.head())
display(pd.DataFrame([best_decoder_anchor]))


0:	learn: 1.7627753	test: 1.9571867	best: 1.9571867 (0)	total: 246ms	remaining: 6m 8s
200:	learn: 1.3784529	test: 1.5133597	best: 1.5133597 (200)	total: 20.1s	remaining: 2m 9s


KeyboardInterrupt: 

## 09. Poisson Variant

Di variant ini hanya objective untuk direct goal models yang diubah menjadi **Poisson**.  
Tujuannya adalah melihat apakah raw signal count untuk goals jadi lebih baik tanpa mengubah pipeline lain.


In [ ]:
poisson_bundle = train_variant_bundle(
    train_df=train_feature_full,
    feature_cols=full_feature_cols,
    cat_cols=static_categorical_features,
    goal_params=goal_reg_params_poisson,
    total_params=goal_reg_params_rmse,
    gd_params=goal_reg_params_rmse,
    outcome_params=outcome_clf_params,
    variant_name="poisson",
)

raw_poisson_valid = predict_raw_outputs_for_df(valid_freeze_feature_full, poisson_bundle)
decoder_grid_poisson, best_decoder_poisson, pred_poisson_valid = tune_decoder_from_raw_outputs(
    raw_df=raw_poisson_valid,
    decoder_grid=DEFAULT_DECODER_GRID,
    scoreline_prior=scoreline_prior_lookup,
)
valid_pred_match_poisson = raw_poisson_valid.merge(pred_poisson_valid, on="match_id", how="left")
valid_pred_match_poisson.to_csv(PRED_DIR / "valid_pred_match_poisson.csv", index=False)
decoder_grid_poisson.to_csv(SUM_DIR / "decoder_grid_poisson.csv", index=False)

awmae_poisson = awmae_score(
    valid_pred_match_poisson["actual_team_a_goals"],
    valid_pred_match_poisson["actual_team_b_goals"],
    valid_pred_match_poisson["pred_team_a_goals"],
    valid_pred_match_poisson["pred_team_b_goals"],
    valid_pred_match_poisson["tournament"],
)

print(f"[RESULT] poisson valid AW-MAE = {awmae_poisson:.6f}")
display(valid_pred_match_poisson.head())
display(pd.DataFrame([best_decoder_poisson]))


## 10. Tweedie Variant

Variant ini menguji objective **Tweedie** pada goal models.

Untuk menjaga eksperimen tetap fokus dan runtime tetap aman, kita memakai **variance power yang konservatif**:

- `variance_power = 1.3`

Jadi ini bukan brute-force grid, melainkan satu pilihan aman yang masih relevan untuk target count yang skewed.


In [ ]:
tweedie_bundle = train_variant_bundle(
    train_df=train_feature_full,
    feature_cols=full_feature_cols,
    cat_cols=static_categorical_features,
    goal_params=goal_reg_params_tweedie,
    total_params=goal_reg_params_rmse,
    gd_params=goal_reg_params_rmse,
    outcome_params=outcome_clf_params,
    variant_name="tweedie",
)

raw_tweedie_valid = predict_raw_outputs_for_df(valid_freeze_feature_full, tweedie_bundle)
decoder_grid_tweedie, best_decoder_tweedie, pred_tweedie_valid = tune_decoder_from_raw_outputs(
    raw_df=raw_tweedie_valid,
    decoder_grid=DEFAULT_DECODER_GRID,
    scoreline_prior=scoreline_prior_lookup,
)
valid_pred_match_tweedie = raw_tweedie_valid.merge(pred_tweedie_valid, on="match_id", how="left")
valid_pred_match_tweedie.to_csv(PRED_DIR / "valid_pred_match_tweedie.csv", index=False)
decoder_grid_tweedie.to_csv(SUM_DIR / "decoder_grid_tweedie.csv", index=False)

awmae_tweedie = awmae_score(
    valid_pred_match_tweedie["actual_team_a_goals"],
    valid_pred_match_tweedie["actual_team_b_goals"],
    valid_pred_match_tweedie["pred_team_a_goals"],
    valid_pred_match_tweedie["pred_team_b_goals"],
    valid_pred_match_tweedie["tournament"],
)

print(f"[RESULT] tweedie valid AW-MAE = {awmae_tweedie:.6f}")
display(valid_pred_match_tweedie.head())
display(pd.DataFrame([best_decoder_tweedie]))


## 11. Tail Auxiliary Models

Di tahap ini kita melatih classifier tambahan untuk mendeteksi sinyal tail:

- `is_big_margin_5plus`
- `is_big_margin_7plus`
- `is_high_total_6plus`
- `is_high_total_8plus`

Target tambahan ini dipakai bukan untuk menggantikan score prediction, tetapi untuk memberi decoder sinyal bahwa sebuah match mungkin berada di rezim yang tidak moderat.


In [ ]:
tail_models = train_tail_models(
    train_df=train_feature_full,
    feature_cols=full_feature_cols,
    cat_cols=static_categorical_features,
)

valid_tail_probs = predict_tail_probabilities_for_df(valid_freeze_feature_full, tail_models)
display(valid_tail_probs.head())

tail_prob_summary = valid_tail_probs.drop(columns=["match_id"]).describe().T
display(tail_prob_summary)


## 12. Tail-Aware Decoder

Decoder tail-aware dibangun di atas **best objective variant** dari V0 / V1 / V2.

Decoder ini tetap memakai komponen anchor:

- direct goals
- total
- goal difference
- outcome probability
- scoreline prior

Lalu ditambah komponen tail:

- `w_bm5`
- `w_bm7`
- `w_ht`

Intuisinya:

- kandidat skor besar harus mendapat **support** dari tail probabilities,
- kalau support tail lemah, kandidat ekstrem akan menerima penalti tambahan.


In [ ]:
objective_variant_table = pd.DataFrame([
    {"variant_name": "anchor_rmse", "valid_awmae": awmae_anchor},
    {"variant_name": "poisson", "valid_awmae": awmae_poisson},
    {"variant_name": "tweedie", "valid_awmae": awmae_tweedie},
]).sort_values("valid_awmae").reset_index(drop=True)

best_objective_variant_name = objective_variant_table.iloc[0]["variant_name"]
best_objective_raw_valid = {
    "anchor_rmse": raw_anchor_valid,
    "poisson": raw_poisson_valid,
    "tweedie": raw_tweedie_valid,
}[best_objective_variant_name]

TAIL_DECODER_GRID = {
    "MAX_GOALS": [7, 8, 10],
    "w_direct": [0.5, 1.0],
    "w_total": [1.0, 1.5],
    "w_gd": [1.0, 1.5],
    "w_outcome": [1.5, 2.0],
    "w_prior": [0.2, 0.5],
    "w_bm5": [0.0, 0.5, 1.0],
    "w_bm7": [0.0, 0.5, 1.0],
    "w_ht": [0.0, 0.5, 1.0],
}

decoder_grid_tail_aware, best_decoder_tail_aware, pred_tail_aware_valid = tune_tail_aware_decoder_from_raw_outputs(
    raw_df=best_objective_raw_valid,
    tail_prob_df=valid_tail_probs,
    decoder_grid=TAIL_DECODER_GRID,
    scoreline_prior=scoreline_prior_lookup,
)
valid_pred_match_tail_aware = best_objective_raw_valid.merge(pred_tail_aware_valid, on="match_id", how="left")
valid_pred_match_tail_aware = valid_pred_match_tail_aware.merge(valid_tail_probs, on="match_id", how="left")

valid_pred_match_tail_aware.to_csv(PRED_DIR / "valid_pred_match_tail_aware.csv", index=False)
decoder_grid_tail_aware.to_csv(SUM_DIR / "decoder_grid_tail_aware.csv", index=False)

awmae_tail_aware = awmae_score(
    valid_pred_match_tail_aware["actual_team_a_goals"],
    valid_pred_match_tail_aware["actual_team_b_goals"],
    valid_pred_match_tail_aware["pred_team_a_goals"],
    valid_pred_match_tail_aware["pred_team_b_goals"],
    valid_pred_match_tail_aware["tournament"],
)

print(f"best objective variant      : {best_objective_variant_name}")
print(f"[RESULT] tail_aware AW-MAE  : {awmae_tail_aware:.6f}")
display(objective_variant_table)
display(pd.DataFrame([best_decoder_tail_aware]))
display(valid_pred_match_tail_aware.head())


## 13. Variant Comparison Table

Tabel ini menjadi ringkasan utama untuk melihat:

- objective mana yang terbaik secara validation fair,
- apakah tail-aware decoder memberi tambahan value,
- dan variant mana yang layak dijadikan **best safe pipeline**.


In [ ]:
variant_metrics = pd.DataFrame([
    {
        "variant_name": "anchor_rmse",
        "goal_objective": "RMSE",
        "tail_aware": False,
        "valid_awmae": float(awmae_anchor),
        "notes": "history-full + freeze-safe anchor",
    },
    {
        "variant_name": "poisson",
        "goal_objective": "Poisson",
        "tail_aware": False,
        "valid_awmae": float(awmae_poisson),
        "notes": "goal_a/goal_b memakai Poisson",
    },
    {
        "variant_name": "tweedie",
        "goal_objective": f"Tweedie(vp={TWEEDIE_VARIANCE_POWER})",
        "tail_aware": False,
        "valid_awmae": float(awmae_tweedie),
        "notes": "goal_a/goal_b memakai Tweedie",
    },
    {
        "variant_name": "tail_aware_best",
        "goal_objective": best_objective_variant_name,
        "tail_aware": True,
        "valid_awmae": float(awmae_tail_aware),
        "notes": "best objective variant + tail-aware decoder",
    },
]).sort_values("valid_awmae").reset_index(drop=True)

variant_metrics.to_csv(SUM_DIR / "variant_metrics.csv", index=False)
display(variant_metrics)

plt.figure(figsize=(8, 4.5))
sns.barplot(data=variant_metrics, x="variant_name", y="valid_awmae")
plt.title("Validation AW-MAE per Variant")
plt.xlabel("")
plt.ylabel("AW-MAE")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "awmae_variant_comparison.png", dpi=160, bbox_inches="tight")
plt.show()


## 14. Raw Signal Analysis

Sebelum melihat keputusan akhir, kita bandingkan dulu **raw prediction quality** dari objective variants.

Tujuannya penting:  
kalau objective baru memang membantu, semestinya ada sinyal pada:

- outcome accuracy,
- MAE goal_a,
- MAE goal_b,
- MAE total,
- MAE gd.


In [ ]:
def summarize_raw_signal(raw_df: pd.DataFrame, variant_name: str) -> dict:
    outcome_pred = raw_df[["pred_outcome_proba_0", "pred_outcome_proba_1", "pred_outcome_proba_2"]].values.argmax(axis=1)
    outcome_true = build_outcome_target(raw_df["actual_team_a_goals"], raw_df["actual_team_b_goals"]).values

    return {
        "variant_name": variant_name,
        "outcome_accuracy": accuracy_score(outcome_true, outcome_pred),
        "mae_goal_a": mean_absolute_error(raw_df["actual_team_a_goals"], raw_df["pred_goal_a_cont"]),
        "mae_goal_b": mean_absolute_error(raw_df["actual_team_b_goals"], raw_df["pred_goal_b_cont"]),
        "mae_total": mean_absolute_error(raw_df["actual_team_a_goals"] + raw_df["actual_team_b_goals"], raw_df["pred_total_cont"]),
        "mae_gd": mean_absolute_error(raw_df["actual_team_a_goals"] - raw_df["actual_team_b_goals"], raw_df["pred_gd_cont"]),
    }

raw_signal_summary = pd.DataFrame([
    summarize_raw_signal(raw_anchor_valid, "anchor_rmse"),
    summarize_raw_signal(raw_poisson_valid, "poisson"),
    summarize_raw_signal(raw_tweedie_valid, "tweedie"),
]).sort_values("variant_name").reset_index(drop=True)

display(raw_signal_summary)


## 15. Tail Subgroup Analysis

Ini bagian wajib dari EXP04A.

Kita membandingkan AW-MAE khusus pada subgroup:

- `extreme_match`
- `non_extreme_match`

Definisi extreme di sini:

- `max(actual_goals_a, actual_goals_b) >= 6`
  **atau**
- `abs(actual_goal_diff) >= 5`

Kalau tail-aware decoder memang bekerja, efeknya paling jelas harus terlihat di subgroup ini.


In [ ]:
def add_loss_columns(pred_df: pd.DataFrame) -> pd.DataFrame:
    out = pred_df.copy()
    out["match_loss"] = [
        official_match_loss(a, b, c, d)
        for a, b, c, d in zip(
            out["actual_team_a_goals"],
            out["actual_team_b_goals"],
            out["pred_team_a_goals"],
            out["pred_team_b_goals"],
        )
    ]
    out["weight"] = out["tournament"].apply(get_tournament_weight)
    out["is_extreme"] = (
        (out[["actual_team_a_goals", "actual_team_b_goals"]].max(axis=1) >= 6)
        | ((out["actual_team_a_goals"] - out["actual_team_b_goals"]).abs() >= 5)
    ).astype(int)
    return out


def subgroup_awmae(df: pd.DataFrame) -> float:
    w = df["weight"].astype(float).to_numpy()
    l = df["match_loss"].astype(float).to_numpy()
    return float(np.sum(w * l) / np.sum(w)) if np.sum(w) > 0 else np.nan


pred_map_tail = {
    "anchor_rmse": add_loss_columns(valid_pred_match_anchor),
    "poisson": add_loss_columns(valid_pred_match_poisson),
    "tweedie": add_loss_columns(valid_pred_match_tweedie),
    "tail_aware_best": add_loss_columns(valid_pred_match_tail_aware),
}

tail_rows = []
for variant_name, dfp in pred_map_tail.items():
    for ext_val, grp in dfp.groupby("is_extreme"):
        tail_rows.append({
            "variant_name": variant_name,
            "subgroup": "extreme_match" if ext_val == 1 else "non_extreme_match",
            "n_matches": int(len(grp)),
            "awmae": subgroup_awmae(grp),
        })

tail_subgroup_df = pd.DataFrame(tail_rows)
display(tail_subgroup_df)

plt.figure(figsize=(8, 4.5))
sns.barplot(data=tail_subgroup_df, x="variant_name", y="awmae", hue="subgroup")
plt.title("AW-MAE pada Extreme vs Non-Extreme Match")
plt.xlabel("")
plt.ylabel("AW-MAE")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "tail_subgroup_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

top_loss_cases = pred_map_tail["tail_aware_best"].sort_values("match_loss", ascending=False).head(15).copy()
top_loss_cases["actual_score"] = top_loss_cases["actual_team_a_goals"].astype(int).astype(str) + "-" + top_loss_cases["actual_team_b_goals"].astype(int).astype(str)
top_loss_cases["pred_score"] = top_loss_cases["pred_team_a_goals"].astype(int).astype(str) + "-" + top_loss_cases["pred_team_b_goals"].astype(int).astype(str)

display(top_loss_cases[[
    "match_id", "tournament", "actual_score", "pred_score",
    "pred_goal_a_cont", "pred_goal_b_cont", "match_loss", "is_extreme"
]])

fig, ax = plt.subplots(figsize=(16, 6))
ax.axis("off")
tbl = ax.table(
    cellText=top_loss_cases[["match_id", "actual_score", "pred_score", "match_loss", "is_extreme"]].values,
    colLabels=["match_id", "actual", "pred", "match_loss", "is_extreme"],
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.4)
plt.title("Top Loss Cases — Tail-Aware Best")
plt.tight_layout()
plt.savefig(FIG_DIR / "top_loss_case_examples.png", dpi=160, bbox_inches="tight")
plt.show()


## 16. Additional Subgroup Analysis

Setelah melihat subgroup tail, kita juga perlu memastikan perubahan ini tidak diam-diam merusak subgroup lain.

Karena itu kita cek AW-MAE per:

- `gender`
- `neutral`
- top tournament


In [ ]:
base_group_info = valid_fold_base[["match_id", "gender", "neutral", "tournament"]].copy()

subgroup_rows = []
for variant_name, pred_df in pred_map_tail.items():
    tmp = pred_df.merge(base_group_info, on="match_id", how="left", suffixes=("", "_base"))

    for gender, grp in tmp.groupby("gender"):
        subgroup_rows.append({
            "variant_name": variant_name,
            "group_type": "gender",
            "group_value": gender,
            "awmae_subgroup": subgroup_awmae(grp),
            "n_matches": len(grp),
        })

    for neutral, grp in tmp.groupby("neutral"):
        subgroup_rows.append({
            "variant_name": variant_name,
            "group_type": "neutral",
            "group_value": str(neutral),
            "awmae_subgroup": subgroup_awmae(grp),
            "n_matches": len(grp),
        })

    top_tournaments = tmp["tournament"].value_counts().head(8).index.tolist()
    for tournament_name in top_tournaments:
        grp = tmp[tmp["tournament"] == tournament_name]
        subgroup_rows.append({
            "variant_name": variant_name,
            "group_type": "tournament",
            "group_value": tournament_name,
            "awmae_subgroup": subgroup_awmae(grp),
            "n_matches": len(grp),
        })

subgroup_df = pd.DataFrame(subgroup_rows)
display(subgroup_df.head(30))

for group_type, fname, title in [
    ("gender", "subgroup_awmae_gender.png", "AW-MAE per Gender"),
    ("neutral", "subgroup_awmae_neutral.png", "AW-MAE per Neutral"),
]:
    plot_df = subgroup_df[subgroup_df["group_type"] == group_type].copy()
    plt.figure(figsize=(9, 4.5))
    sns.barplot(data=plot_df, x="variant_name", y="awmae_subgroup", hue="group_value")
    plt.title(title)
    plt.xlabel("")
    plt.ylabel("AW-MAE")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=160, bbox_inches="tight")
    plt.show()

plot_tourn = subgroup_df[subgroup_df["group_type"] == "tournament"].copy()
if not plot_tourn.empty:
    plt.figure(figsize=(12, 5))
    sns.barplot(data=plot_tourn, x="group_value", y="awmae_subgroup", hue="variant_name")
    plt.title("AW-MAE pada Top Tournament")
    plt.xlabel("Tournament")
    plt.ylabel("AW-MAE")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "subgroup_awmae_top_tournament.png", dpi=160, bbox_inches="tight")
    plt.show()


## 17. Feature Importance Analysis

Di tahap ini kita lihat tiga sudut penting:

1. feature importance dari **anchor goal model**,
2. feature importance dari **best objective variant goal model**,
3. feature importance dari **tail classifier**.

Tujuannya adalah mengecek apakah:

- history features tetap dominan,
- gap / strength features masih kuat,
- dan sinyal tail memang ditangkap oleh feature-feature yang masuk akal.


In [ ]:
def extract_feature_importance_from_model(model, feature_cols: list[str]) -> pd.DataFrame:
    imp = model.get_feature_importance()
    return pd.DataFrame({"feature": feature_cols, "importance": imp}).sort_values("importance", ascending=False)

best_objective_bundle = {
    "anchor_rmse": anchor_bundle,
    "poisson": poisson_bundle,
    "tweedie": tweedie_bundle,
}[best_objective_variant_name]

fi_anchor_goal = extract_feature_importance_from_model(anchor_bundle["goal_a"], full_feature_cols).head(20)
fi_best_goal = extract_feature_importance_from_model(best_objective_bundle["goal_a"], full_feature_cols).head(20)
fi_tail = extract_feature_importance_from_model(tail_models["clf_big_margin_5plus"], full_feature_cols).head(20)

display(fi_anchor_goal)
display(fi_best_goal)
display(fi_tail)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, dfp, title in [
    (axes[0], fi_anchor_goal, "Anchor Goal-A FI"),
    (axes[1], fi_best_goal, f"Best Objective Goal-A FI ({best_objective_variant_name})"),
    (axes[2], fi_tail, "Tail Classifier FI (big_margin_5plus)"),
]:
    ax.barh(dfp["feature"][::-1], dfp["importance"][::-1])
    ax.set_title(title)
    ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "feature_importance_tail_models.png", dpi=160, bbox_inches="tight")
plt.show()


## 18. Keputusan Eksperimen

Bagian ini harus menghasilkan keputusan yang **tegas**, bukan ambigu.

Pilihan keputusan:

- **A** — Objective/tail refinement membantu secara nyata dan jadi mainline baru.
- **B** — Objective/tail refinement membantu tipis, tapi belum cukup signifikan.
- **C** — Objective/tail refinement tidak membantu dan anchor tetap terbaik.


In [ ]:
best_safe_variant_name = variant_metrics.iloc[0]["variant_name"]
best_valid_awmae = float(variant_metrics.iloc[0]["valid_awmae"])
anchor_gap = float(awmae_anchor - best_valid_awmae)

if best_safe_variant_name == "tail_aware_best" and anchor_gap >= 0.05:
    experiment_decision = "A — Objective/tail refinement membantu secara nyata dan layak jadi mainline baru."
elif best_valid_awmae < awmae_anchor:
    experiment_decision = "B — Objective/tail refinement membantu, tetapi selisihnya masih tipis dan perlu validasi lanjutan."
else:
    experiment_decision = "C — Objective/tail refinement belum membantu; anchor tetap terbaik dan EXP04B layak dipertimbangkan."

print(f"Best safe variant : {best_safe_variant_name}")
print(f"Best valid AW-MAE : {best_valid_awmae:.6f}")
print(f"Anchor AW-MAE     : {awmae_anchor:.6f}")
print(f"Gap vs anchor     : {anchor_gap:.6f}")
print(f"Decision          : {experiment_decision}")


## 19. Retrain Best Safe Variant pada Full Train

Setelah selection fair di validation selesai, barulah variant terbaik di-retrain pada seluruh train.

Kalau variant terbaik adalah `tail_aware_best`, maka yang di-retrain adalah:

- objective bundle terbaiknya, dan
- tail auxiliary classifiers.


In [ ]:
full_train_history_full, cutoff_team_states_test, cutoff_h2h_states_test = build_train_history_features_full(train_match_base)

full_train_feature = (
    train_match_base
    .merge(train_static_df.drop(columns=base_cols_to_drop_from_static, errors="ignore"), on="match_id", how="left")
    .merge(full_train_history_full, on="match_id", how="left")
)

full_train_feature["y_goal_a"] = full_train_feature["team_a_goals"].astype(float)
full_train_feature["y_goal_b"] = full_train_feature["team_b_goals"].astype(float)
full_train_feature["y_total"] = full_train_feature["team_a_goals"].astype(float) + full_train_feature["team_b_goals"].astype(float)
full_train_feature["y_gd"] = full_train_feature["team_a_goals"].astype(float) - full_train_feature["team_b_goals"].astype(float)
full_train_feature["y_outcome"] = build_outcome_target(full_train_feature["team_a_goals"], full_train_feature["team_b_goals"]).astype(int)
full_train_feature["is_big_margin_5plus"] = (full_train_feature["y_gd"].abs() >= 5).astype(int)
full_train_feature["is_big_margin_7plus"] = (full_train_feature["y_gd"].abs() >= 7).astype(int)
full_train_feature["is_high_total_6plus"] = (full_train_feature["y_total"] >= 6).astype(int)
full_train_feature["is_high_total_8plus"] = (full_train_feature["y_total"] >= 8).astype(int)

best_objective_name_for_retrain = best_objective_variant_name if best_safe_variant_name == "tail_aware_best" else best_safe_variant_name

objective_param_lookup = {
    "anchor_rmse": goal_reg_params_rmse,
    "poisson": goal_reg_params_poisson,
    "tweedie": goal_reg_params_tweedie,
}
retrain_bundle = train_variant_bundle(
    train_df=full_train_feature,
    feature_cols=full_feature_cols,
    cat_cols=static_categorical_features,
    goal_params=objective_param_lookup[best_objective_name_for_retrain],
    total_params=goal_reg_params_rmse,
    gd_params=goal_reg_params_rmse,
    outcome_params=outcome_clf_params,
    variant_name=best_objective_name_for_retrain,
)

retrain_tail_models = None
if best_safe_variant_name == "tail_aware_best":
    retrain_tail_models = train_tail_models(
        train_df=full_train_feature,
        feature_cols=full_feature_cols,
        cat_cols=static_categorical_features,
    )

print(f"[OK] Retrain selesai untuk best_safe_variant = {best_safe_variant_name}")


## 20. Test Inference

Tahap ini menjalankan freeze-safe inference pada test.

Karena eksperimen ini fokus pada **standalone safe pipeline**, kita hanya memakai:

- state hasil train penuh,
- static test features,
- history snapshot freeze-safe,
- dan decoder terbaik dari validation.


In [ ]:
test_freeze_feature_full = simulate_freeze_feature_rows(
    future_match_df=test_match_base,
    static_feature_df=test_static_df,
    team_states_at_cutoff=cutoff_team_states_test,
    h2h_states_at_cutoff=cutoff_h2h_states_test,
)

test_raw_outputs = predict_raw_outputs_for_df(test_freeze_feature_full, retrain_bundle)
test_match_lookup = test_match_base.set_index("match_id")[["team_a", "team_b"]]

if best_safe_variant_name == "tail_aware_best":
    test_tail_probs = predict_tail_probabilities_for_df(test_freeze_feature_full, retrain_tail_models)
    test_decode_input = test_raw_outputs.merge(test_tail_probs, on="match_id", how="left")
    best_prior_test = scoreline_prior_lookup[int(best_decoder_tail_aware["MAX_GOALS"])]

    pred_rows = []
    for _, row in tqdm(test_decode_input.iterrows(), total=len(test_decode_input), desc="decode_test_tail_aware"):
        pair = decode_single_match_tail_aware(
            raw_row=row,
            tail_row=row,
            decoder_params=best_decoder_tail_aware,
            scoreline_prior=best_prior_test,
        )
        pred_rows.append({
            "match_id": row["match_id"],
            "team_a": test_match_lookup.loc[row["match_id"], "team_a"],
            "team_b": test_match_lookup.loc[row["match_id"], "team_b"],
            "pred_team_a_goals": int(pair[0]),
            "pred_team_b_goals": int(pair[1]),
        })
else:
    decoder_lookup = {
        "anchor_rmse": best_decoder_anchor,
        "poisson": best_decoder_poisson,
        "tweedie": best_decoder_tweedie,
    }
    selected_decoder = decoder_lookup[best_safe_variant_name]
    best_prior_test = scoreline_prior_lookup[int(selected_decoder["MAX_GOALS"])]

    pred_rows = []
    for _, row in tqdm(test_raw_outputs.iterrows(), total=len(test_raw_outputs), desc="decode_test_anchor_style"):
        pair = decode_single_match_score(
            pred_goal_a=float(row["pred_goal_a_cont"]),
            pred_goal_b=float(row["pred_goal_b_cont"]),
            pred_total=float(row["pred_total_cont"]),
            pred_gd=float(row["pred_gd_cont"]),
            pred_outcome_proba=np.array(
                [row["pred_outcome_proba_0"], row["pred_outcome_proba_1"], row["pred_outcome_proba_2"]],
                dtype=float,
            ),
            scoreline_prior=best_prior_test,
            max_goals=int(selected_decoder["MAX_GOALS"]),
            w_direct=float(selected_decoder["w_direct"]),
            w_total=float(selected_decoder["w_total"]),
            w_gd=float(selected_decoder["w_gd"]),
            w_outcome=float(selected_decoder["w_outcome"]),
            w_prior=float(selected_decoder["w_prior"]),
        )
        pred_rows.append({
            "match_id": row["match_id"],
            "team_a": test_match_lookup.loc[row["match_id"], "team_a"],
            "team_b": test_match_lookup.loc[row["match_id"], "team_b"],
            "pred_team_a_goals": int(pair[0]),
            "pred_team_b_goals": int(pair[1]),
        })

test_pred_match_best_safe = pd.DataFrame(pred_rows).sort_values("match_id").reset_index(drop=True)
test_pred_match_best_safe.to_csv(PRED_DIR / "test_pred_match_best_safe.csv", index=False)

print(f"test_pred_match_best_safe shape : {test_pred_match_best_safe.shape}")
display(test_pred_match_best_safe.head())


## 21. Reverse Mapping ke Submission

Prediksi match-level dikembalikan ke format row-level submission, lalu diverifikasi agar:

- jumlah row sama,
- urutan `Id` sama,
- kolom sesuai,
- dan tidak ada missing value.


In [ ]:
submission = match_predictions_to_submission(
    test_row_df=test_clean,
    pred_match_df=test_pred_match_best_safe,
)

assert len(submission) == len(sample_sub), "Jumlah row submission tidak sama dengan sample submission."
assert (submission["Id"].values == sample_sub["Id"].values).all(), "Urutan Id tidak sama dengan sample submission."
assert list(submission.columns) == ["Id", "team_goals", "opp_goals"], "Format kolom submission salah."
assert submission["team_goals"].notna().all(), "Masih ada missing team_goals."
assert submission["opp_goals"].notna().all(), "Masih ada missing opp_goals."

submission["team_goals"] = submission["team_goals"].astype(int)
submission["opp_goals"] = submission["opp_goals"].astype(int)

submission.to_csv(SUB_DIR / "submission_exp04a_best_safe.csv", index=False)

print(f"[OK] submission shape = {submission.shape}")
display(submission.head())


## 22. Ringkasan Hasil Eksperimen

Bagian akhir ini harus menjawab empat pertanyaan utama:

1. objective mana yang terbaik?
2. apakah tail-aware decoder membantu?
3. apakah subgroup extreme membaik?
4. apakah pipeline terbaik menyalip baseline history-full sebelumnya?

Keputusan final tetap harus dibaca dari angka validation fair di atas, bukan dari asumsi.


In [ ]:
extreme_cmp = tail_subgroup_df.pivot(index="variant_name", columns="subgroup", values="awmae").reset_index()

summary_payload = {
    "best_objective_variant": best_objective_variant_name,
    "best_safe_variant": best_safe_variant_name,
    "anchor_valid_awmae": float(awmae_anchor),
    "poisson_valid_awmae": float(awmae_poisson),
    "tweedie_valid_awmae": float(awmae_tweedie),
    "tail_aware_valid_awmae": float(awmae_tail_aware),
    "decision": experiment_decision,
}

experiment_notes = []
experiment_notes.append("EXP 04A — CatBoost objective refinement + tail-aware decoder")
experiment_notes.append(f"Best objective variant: {best_objective_variant_name}")
experiment_notes.append(f"Best safe variant: {best_safe_variant_name}")
experiment_notes.append(f"Anchor RMSE valid AW-MAE: {awmae_anchor:.6f}")
experiment_notes.append(f"Poisson valid AW-MAE: {awmae_poisson:.6f}")
experiment_notes.append(f"Tweedie valid AW-MAE: {awmae_tweedie:.6f}")
experiment_notes.append(f"Tail-aware valid AW-MAE: {awmae_tail_aware:.6f}")
experiment_notes.append(f"Decision: {experiment_decision}")

with open(SUM_DIR / "experiment_notes.txt", "w", encoding="utf-8") as fp:
    fp.write("\n".join(experiment_notes))

print("\n".join(experiment_notes))
display(variant_metrics)
display(raw_signal_summary)
display(extreme_cmp)
display(pd.Series(summary_payload))
